# 2. risk_factors_cervical_cancer.csv

In [ ]:
# ============================================================
# MPRF — Risk Factors Cervical Cancer Dataset
# Score combiné : α·F1_min + (1-α)·G-mean
# Métriques complètes + Tests statistiques
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
df = pd.read_csv("1. risk_factors_cervical_cancer.csv")
df.replace("?", np.nan, inplace=True)
df = df.apply(pd.to_numeric, errors="coerce")
df.fillna(df.median(numeric_only=True), inplace=True)

TARGET = "Biopsy"
feature_cols = [c for c in df.columns if c != TARGET]

X = df[feature_cols].values
y = df[TARGET].values.astype(int)

counts = np.bincount(y)
print(f"Shape : {X.shape}  |  Classes : {counts}  |  IR ≈ {counts[0]/counts[1]:.1f}:1")

K_BIRF = 10

# ┌─────────────────────────────────────────────────────────┐
# │  ALPHA — paramètre du score combiné MPRF               │
# │  Score(S) = α·F1_min(S) + (1-α)·G-mean(S)             │
# │  α = 1.0  → pur F1_min  (comportement original)        │
# │  α = 0.5  → équilibre F1_min et G-mean                 │
# │  α = 0.0  → pur G-mean                                 │
# └─────────────────────────────────────────────────────────┘
ALPHA = 0.6   # ← MODIFIER ICI

print(f"Score combiné : α={ALPHA:.1f} · F1_min + {1-ALPHA:.1f} · G-mean")

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)

    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

# ── Vote majoritaire ─────────────────────────────────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

# ── Score combiné pour la sélection ──────────────────────────
def combined_score(y_true, y_pred, alpha):
    """Score(S) = α·F1_min(S) + (1-α)·G-mean(S)"""
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    f1  = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm  = np.sqrt(recall_1 * recall_0)

    return alpha * f1 + (1 - alpha) * gm

# ── MPRF — Forward Selection hill climbing ───────────────────
def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    """
    Hill climbing forward selection :
    1. Init : un arbre tiré aléatoirement (seed propre au fold)
    2. Critère : Score(S) = α·F1_min(S) + (1-α)·G-mean(S)
    3. À chaque étape : on évalue TOUS les candidats restants
       et on ajoute le meilleur même si égalité (plateau)
    4. best_score mis à jour UNIQUEMENT si amélioration stricte
    5. On s'arrête si tous les candidats dégradent ou si k atteint
    """
    n = len(trees)

    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)

    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)

    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected),
                                alpha)

    while len(selected) < k and remaining:

        best_cand       = None
        best_cand_score = -np.inf

        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score = score
                best_cand       = idx

        # Avancer si meilleur OU égal (plateau accepté)
        # best_score mis à jour UNIQUEMENT si amélioration stricte
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break

    return selected

# ── OBP ──────────────────────────────────────────────────────
def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

# ── IBEP ─────────────────────────────────────────────────────
def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]

# ── Cliff's Delta ─────────────────────────────────────────────
def cliffs_delta(x, y):
    x, y = np.array(x), np.array(y)
    n, m = len(x), len(y)
    more = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    delta = (more - less) / (n * m)
    abs_d = abs(delta)
    if abs_d < 0.147:
        magnitude = "negligible"
    elif abs_d < 0.330:
        magnitude = "small"
    elif abs_d < 0.474:
        magnitude = "medium"
    else:
        magnitude = "large"
    return delta, magnitude

# ── display_results ──────────────────────────────────────────
def display_results(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    print(f"\n{'='*85}")
    print(f"  Résultats — {dataset_label}")
    print(f"{'='*85}")
    scores = {}
    for method in methods:
        vals = results.get(f"{method}_f1_min", [np.nan])
        scores[method] = np.mean(vals)
        size = np.mean(results.get(f"{method}_size",      [np.nan]))
        f1   = np.mean(results.get(f"{method}_f1_min",    [np.nan]))
        gm   = np.mean(results.get(f"{method}_gmean",     [np.nan]))
        aup  = np.mean(results.get(f"{method}_auprc",     [np.nan]))
        prec = np.mean(results.get(f"{method}_precision", [np.nan]))
        rec  = np.mean(results.get(f"{method}_recall",    [np.nan]))
        print(f"  {method:<6} | taille={size:5.1f} | "
              f"F1_min={f1:.3f} | G-mean={gm:.3f} | "
              f"AUPRC={aup:.3f} | Prec={prec:.3f} | Recall={rec:.3f}")
    winner = max(scores, key=scores.get)
    print(f"\n🏆 Gagnant (F1 minoritaire) : {winner} = {scores[winner]:.3f}")
    print(f"{'='*85}")

# ── Tests statistiques ────────────────────────────────────────
def statistical_analysis(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    metrics = ["f1_min", "gmean", "auprc"]
    ref     = "MPRF"

    print(f"\n{'='*75}")
    print(f"  Tests statistiques — {dataset_label}")
    print(f"{'='*75}")

    for metric in metrics:
        print(f"\n  Métrique : {metric.upper()}")
        print(f"  {'Comparaison':<20} {'p-value':>10} {'Significatif':>14} "
              f"{'Cliff δ':>10} {'Magnitude':>12}")
        print(f"  {'-'*66}")

        ref_vals = np.array(results.get(f"{ref}_{metric}", [np.nan]))

        for method in methods:
            if method == ref:
                continue
            comp_vals = np.array(results.get(f"{method}_{metric}", [np.nan]))
            try:
                if np.all(ref_vals == comp_vals):
                    p_val = 1.0
                else:
                    _, p_val = wilcoxon(ref_vals, comp_vals)
            except Exception:
                p_val = np.nan

            sig   = "✅ Oui" if (not np.isnan(p_val) and p_val < 0.05) else "❌ Non"
            delta, magnitude = cliffs_delta(ref_vals, comp_vals)

            print(f"  {ref} vs {method:<14} {p_val:>10.4f} {sig:>14} "
                  f"{delta:>10.3f} {magnitude:>12}")

    print(f"\n  Seuil α = 0.05 | "
          f"Cliff δ : negligible<0.147, small<0.330, medium<0.474, large≥0.474")
    print(f"{'='*75}")

# ── run_experiment_v2 ─────────────────────────────────────────
def run_experiment_v2(X, y, dataset_name, n_trees=50, k=K_BIRF, n_splits=10):

    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = defaultdict(list)

    print(f"\n{'='*60}")
    print(f"Dataset : {dataset_name}")
    print(f"RF balanced : {n_trees} arbres | MPRF : k={k} | Folds : {n_splits}")
    print(f"Score : α={ALPHA:.1f}·F1_min + {1-ALPHA:.1f}·G-mean")
    print(f"OBP et IBEP : taille automatique")
    print(f"{'='*60}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)

        X_val       = X_train[val_idx]
        X_train_sub = X_train[train_sub_idx]
        y_val       = y_train[val_idx]
        y_train_sub = y_train[train_sub_idx]

        rf = RandomForestClassifier(
            n_estimators=n_trees,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        # BRF
        y_pred_brf  = rf.predict(X_test)
        y_proba_brf = rf.predict_proba(X_test)[:, 1]
        for k_m, v in compute_metrics(y_test, y_pred_brf, y_proba_brf).items():
            results[f'BRF_{k_m}'].append(v)
        results['BRF_size'].append(n_trees)

        # MPRF
        selected_mprf = MPRF(trees, X_val, y_val, k, fold, ALPHA)
        y_pred_mprf   = ensemble_predict(trees, X_test, selected_mprf)
        y_proba_mprf  = ensemble_proba(trees, X_test, selected_mprf)
        for k_m, v in compute_metrics(y_test, y_pred_mprf, y_proba_mprf).items():
            results[f'MPRF_{k_m}'].append(v)
        results['MPRF_size'].append(len(selected_mprf))

        # OBP
        selected_obp  = OBP(trees, X_val, y_val, n_trees)
        y_pred_obp    = ensemble_predict(trees, X_test, selected_obp)
        y_proba_obp   = ensemble_proba(trees, X_test, selected_obp)
        for k_m, v in compute_metrics(y_test, y_pred_obp, y_proba_obp).items():
            results[f'OBP_{k_m}'].append(v)
        results['OBP_size'].append(len(selected_obp))

        # IBEP
        selected_ibep = IBEP(trees, X_val, y_val, n_trees)
        y_pred_ibep   = ensemble_predict(trees, X_test, selected_ibep)
        y_proba_ibep  = ensemble_proba(trees, X_test, selected_ibep)
        for k_m, v in compute_metrics(y_test, y_pred_ibep, y_proba_ibep).items():
            results[f'IBEP_{k_m}'].append(v)
        results['IBEP_size'].append(len(selected_ibep))

        print(f"✅ MPRF({len(selected_mprf)})={np.mean(results['MPRF_f1_min']):.3f} | "
              f"OBP({len(selected_obp)})={np.mean(results['OBP_f1_min']):.3f} | "
              f"IBEP({len(selected_ibep)})={np.mean(results['IBEP_f1_min']):.3f} | "
              f"BRF({n_trees})={np.mean(results['BRF_f1_min']):.3f}")

    return results

# ── Lancement ────────────────────────────────────────────────
results_cc = run_experiment_v2(
    X, y, "Risk Factors Cervical Cancer (Biopsy)",
    n_trees=50, k=K_BIRF, n_splits=10
)

display_results(results_cc,
                "Risk Factors Cervical Cancer — Biopsy (minoritaire = 1)")
statistical_analysis(results_cc,
                     "Risk Factors Cervical Cancer — Biopsy (minoritaire = 1)")

# ── Tableau fold par fold ─────────────────────────────────────
print(f"\n{'='*60}")
print("  Valeurs fold par fold — F1_min")
print(f"{'='*60}")
methods_list = ["BRF", "MPRF", "OBP", "IBEP"]
print(f"{'Fold':<6} " + " ".join(f"{m:>8}" for m in methods_list))
print("-" * 42)
for i in range(10):
    row = f"{i+1:<6} "
    for m in methods_list:
        val = results_cc[f"{m}_f1_min"][i]
        row += f"  {val:.3f}  "
    print(row)
print("-" * 42)
print(f"{'Moy':<6} ", end="")
for m in methods_list:
    print(f"  {np.mean(results_cc[f'{m}_f1_min']):.3f}  ", end="")
print()
print(f"{'Std':<6} ", end="")
for m in methods_list:
    print(f"  {np.std(results_cc[f'{m}_f1_min']):.3f}  ", end="")
print()

Shape : (858, 35)  |  Classes : [803  55]  |  IR ≈ 14.6:1
Score combiné : α=0.6 · F1_min + 0.4 · G-mean

Dataset : Risk Factors Cervical Cancer (Biopsy)
RF balanced : 50 arbres | MPRF : k=10 | Folds : 10
Score : α=0.6·F1_min + 0.4·G-mean
OBP et IBEP : taille automatique
  Fold 1/10... ✅ MPRF(10)=0.364 | OBP(24)=0.600 | IBEP(7)=0.500 | BRF(50)=0.444
  Fold 2/10... ✅ MPRF(2)=0.382 | OBP(6)=0.600 | IBEP(3)=0.500 | BRF(50)=0.472
  Fold 3/10... ✅ MPRF(2)=0.477 | OBP(2)=0.582 | IBEP(8)=0.533 | BRF(50)=0.463
  Fold 4/10... ✅ MPRF(4)=0.524 | OBP(2)=0.575 | IBEP(5)=0.582 | BRF(50)=0.410
  Fold 5/10... ✅ MPRF(10)=0.519 | OBP(8)=0.614 | IBEP(5)=0.599 | BRF(50)=0.473
  Fold 6/10... ✅ MPRF(5)=0.572 | OBP(16)=0.612 | IBEP(3)=0.599 | BRF(50)=0.494
  Fold 7/10... ✅ MPRF(10)=0.594 | OBP(8)=0.628 | IBEP(3)=0.599 | BRF(50)=0.459
  Fold 8/10... ✅ MPRF(4)=0.611 | OBP(2)=0.646 | IBEP(1)=0.599 | BRF(50)=0.477
  Fold 9/10... ✅ MPRF(4)=0.642 | OBP(12)=0.655 | IBEP(3)=0.607 | BRF(50)=0.513
  Fold 10/10... ✅ MPR

# Hepatisis

In [ ]:
# ============================================================
# MPRF — Hepatitis C Dataset
# Score combiné : α·F1_min + (1-α)·G-mean
# Métriques complètes + Tests statistiques
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
df = pd.read_csv("2. HepatitisCdata.csv")

df["Category"] = df["Category"].astype(str).str.strip()
df["Category"] = df["Category"].map(lambda x:
    0 if "Blood Donor" in x else
    (1 if "Hepatitis" in x else np.nan)
)
df = df.dropna(subset=["Category"])

TARGET = "Category"
drop_cols = [TARGET, "Unnamed: 0"]
feature_cols = [c for c in df.columns if c not in drop_cols]
df_feat = df[feature_cols].copy()

for col in df_feat.columns:
    if df_feat[col].dtype == object:
        le = LabelEncoder()
        mask = df_feat[col].notna()
        df_feat.loc[mask, col] = le.fit_transform(
            df_feat.loc[mask, col].astype(str)
        )
        df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")

df_feat = df_feat.apply(pd.to_numeric, errors="coerce")
df_feat.fillna(df_feat.median(numeric_only=True), inplace=True)

X = df_feat.values
y = df[TARGET].values.astype(int)

counts = np.bincount(y)
print(f"Shape : {X.shape}  |  Classes : {counts}  |  IR ≈ {counts[0]/counts[1]:.1f}:1")

K_BIRF = 10

# ┌─────────────────────────────────────────────────────────┐
# │  ALPHA — paramètre du score combiné MPRF               │
# │  Score(S) = α·F1_min(S) + (1-α)·G-mean(S)             │
# │  α = 1.0  → pur F1_min                                 │
# │  α = 0.7  → recommandé                                 │
# │  α = 0.5  → équilibre F1_min et G-mean                 │
# └─────────────────────────────────────────────────────────┘
ALPHA = 0.6   # ← MODIFIER ICI

print(f"Score combiné : α={ALPHA:.1f} · F1_min + {1-ALPHA:.1f} · G-mean")

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)

    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

# ── Vote majoritaire ─────────────────────────────────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

# ── Score combiné pour la sélection ──────────────────────────
def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)

    return alpha * f1 + (1 - alpha) * gm

# ── MPRF — Forward Selection hill climbing ───────────────────
def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    """
    Hill climbing forward selection :
    1. Init : un arbre tiré aléatoirement (seed propre au fold)
    2. Critère : Score(S) = α·F1_min(S) + (1-α)·G-mean(S)
    3. À chaque étape : on évalue TOUS les candidats restants
       et on ajoute le meilleur même si égalité (plateau)
    4. best_score mis à jour UNIQUEMENT si amélioration stricte
    5. On s'arrête si tous les candidats dégradent ou si k atteint
    """
    n = len(trees)

    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)

    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)

    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected),
                                alpha)

    while len(selected) < k and remaining:

        best_cand       = None
        best_cand_score = -np.inf

        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score = score
                best_cand       = idx

        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break

    return selected

# ── OBP ──────────────────────────────────────────────────────
def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

# ── IBEP ─────────────────────────────────────────────────────
def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]

# ── Cliff's Delta ─────────────────────────────────────────────
def cliffs_delta(x, y):
    x, y = np.array(x), np.array(y)
    n, m = len(x), len(y)
    more = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    delta = (more - less) / (n * m)
    abs_d = abs(delta)
    if abs_d < 0.147:
        magnitude = "negligible"
    elif abs_d < 0.330:
        magnitude = "small"
    elif abs_d < 0.474:
        magnitude = "medium"
    else:
        magnitude = "large"
    return delta, magnitude

# ── display_results ──────────────────────────────────────────
def display_results(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    print(f"\n{'='*85}")
    print(f"  Résultats — {dataset_label}")
    print(f"{'='*85}")
    scores = {}
    for method in methods:
        vals = results.get(f"{method}_f1_min", [np.nan])
        scores[method] = np.mean(vals)
        size = np.mean(results.get(f"{method}_size",      [np.nan]))
        f1   = np.mean(results.get(f"{method}_f1_min",    [np.nan]))
        gm   = np.mean(results.get(f"{method}_gmean",     [np.nan]))
        aup  = np.mean(results.get(f"{method}_auprc",     [np.nan]))
        prec = np.mean(results.get(f"{method}_precision", [np.nan]))
        rec  = np.mean(results.get(f"{method}_recall",    [np.nan]))
        print(f"  {method:<6} | taille={size:5.1f} | "
              f"F1_min={f1:.3f} | G-mean={gm:.3f} | "
              f"AUPRC={aup:.3f} | Prec={prec:.3f} | Recall={rec:.3f}")
    winner = max(scores, key=scores.get)
    print(f"\n🏆 Gagnant (F1 minoritaire) : {winner} = {scores[winner]:.3f}")
    print(f"{'='*85}")

# ── Tests statistiques ────────────────────────────────────────
def statistical_analysis(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    metrics = ["f1_min", "gmean", "auprc"]
    ref     = "MPRF"

    print(f"\n{'='*75}")
    print(f"  Tests statistiques — {dataset_label}")
    print(f"{'='*75}")

    for metric in metrics:
        print(f"\n  Métrique : {metric.upper()}")
        print(f"  {'Comparaison':<20} {'p-value':>10} {'Significatif':>14} "
              f"{'Cliff δ':>10} {'Magnitude':>12}")
        print(f"  {'-'*66}")

        ref_vals = np.array(results.get(f"{ref}_{metric}", [np.nan]))

        for method in methods:
            if method == ref:
                continue
            comp_vals = np.array(results.get(f"{method}_{metric}", [np.nan]))
            try:
                if np.all(ref_vals == comp_vals):
                    p_val = 1.0
                else:
                    _, p_val = wilcoxon(ref_vals, comp_vals)
            except Exception:
                p_val = np.nan

            sig   = "✅ Oui" if (not np.isnan(p_val) and p_val < 0.05) else "❌ Non"
            delta, magnitude = cliffs_delta(ref_vals, comp_vals)

            print(f"  {ref} vs {method:<14} {p_val:>10.4f} {sig:>14} "
                  f"{delta:>10.3f} {magnitude:>12}")

    print(f"\n  Seuil α = 0.05 | "
          f"Cliff δ : negligible<0.147, small<0.330, medium<0.474, large≥0.474")
    print(f"{'='*75}")

# ── run_experiment_v2 ─────────────────────────────────────────
def run_experiment_v2(X, y, dataset_name, n_trees=50, k=K_BIRF, n_splits=10):

    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = defaultdict(list)

    print(f"\n{'='*60}")
    print(f"Dataset : {dataset_name}")
    print(f"RF balanced : {n_trees} arbres | MPRF : k={k} | Folds : {n_splits}")
    print(f"Score : α={ALPHA:.1f}·F1_min + {1-ALPHA:.1f}·G-mean")
    print(f"OBP et IBEP : taille automatique")
    print(f"{'='*60}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)

        X_val       = X_train[val_idx]
        X_train_sub = X_train[train_sub_idx]
        y_val       = y_train[val_idx]
        y_train_sub = y_train[train_sub_idx]

        rf = RandomForestClassifier(
            n_estimators=n_trees,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        # BRF
        y_pred_brf  = rf.predict(X_test)
        y_proba_brf = rf.predict_proba(X_test)[:, 1]
        for k_m, v in compute_metrics(y_test, y_pred_brf, y_proba_brf).items():
            results[f'BRF_{k_m}'].append(v)
        results['BRF_size'].append(n_trees)

        # MPRF
        selected_mprf = MPRF(trees, X_val, y_val, k, fold, ALPHA)
        y_pred_mprf   = ensemble_predict(trees, X_test, selected_mprf)
        y_proba_mprf  = ensemble_proba(trees, X_test, selected_mprf)
        for k_m, v in compute_metrics(y_test, y_pred_mprf, y_proba_mprf).items():
            results[f'MPRF_{k_m}'].append(v)
        results['MPRF_size'].append(len(selected_mprf))

        # OBP
        selected_obp  = OBP(trees, X_val, y_val, n_trees)
        y_pred_obp    = ensemble_predict(trees, X_test, selected_obp)
        y_proba_obp   = ensemble_proba(trees, X_test, selected_obp)
        for k_m, v in compute_metrics(y_test, y_pred_obp, y_proba_obp).items():
            results[f'OBP_{k_m}'].append(v)
        results['OBP_size'].append(len(selected_obp))

        # IBEP
        selected_ibep = IBEP(trees, X_val, y_val, n_trees)
        y_pred_ibep   = ensemble_predict(trees, X_test, selected_ibep)
        y_proba_ibep  = ensemble_proba(trees, X_test, selected_ibep)
        for k_m, v in compute_metrics(y_test, y_pred_ibep, y_proba_ibep).items():
            results[f'IBEP_{k_m}'].append(v)
        results['IBEP_size'].append(len(selected_ibep))

        print(f"✅ MPRF({len(selected_mprf)})={np.mean(results['MPRF_f1_min']):.3f} | "
              f"OBP({len(selected_obp)})={np.mean(results['OBP_f1_min']):.3f} | "
              f"IBEP({len(selected_ibep)})={np.mean(results['IBEP_f1_min']):.3f} | "
              f"BRF({n_trees})={np.mean(results['BRF_f1_min']):.3f}")

    return results

# ── Lancement ────────────────────────────────────────────────
results_hep = run_experiment_v2(
    X, y, "Hepatitis C (Hepatitis vs Blood Donor)",
    n_trees=50, k=K_BIRF, n_splits=10
)

display_results(results_hep,
                "Hepatitis C — Hepatitis=1 vs Blood Donor=0")
statistical_analysis(results_hep,
                     "Hepatitis C — Hepatitis=1 vs Blood Donor=0")

# ── Tableau fold par fold ─────────────────────────────────────
print(f"\n{'='*60}")
print("  Valeurs fold par fold — F1_min")
print(f"{'='*60}")
methods_list = ["BRF", "MPRF", "OBP", "IBEP"]
print(f"{'Fold':<6} " + " ".join(f"{m:>8}" for m in methods_list))
print("-" * 42)
for i in range(10):
    row = f"{i+1:<6} "
    for m in methods_list:
        val = results_hep[f"{m}_f1_min"][i]
        row += f"  {val:.3f}  "
    print(row)
print("-" * 42)
print(f"{'Moy':<6} ", end="")
for m in methods_list:
    print(f"  {np.mean(results_hep[f'{m}_f1_min']):.3f}  ", end="")
print()
print(f"{'Std':<6} ", end="")
for m in methods_list:
    print(f"  {np.std(results_hep[f'{m}_f1_min']):.3f}  ", end="")
print()

Shape : (564, 12)  |  Classes : [540  24]  |  IR ≈ 22.5:1
Score combiné : α=0.6 · F1_min + 0.4 · G-mean

Dataset : Hepatitis C (Hepatitis vs Blood Donor)
RF balanced : 50 arbres | MPRF : k=10 | Folds : 10
Score : α=0.6·F1_min + 0.4·G-mean
OBP et IBEP : taille automatique
  Fold 1/10... ✅ MPRF(10)=0.500 | OBP(12)=0.500 | IBEP(3)=0.400 | BRF(50)=0.500
  Fold 2/10... ✅ MPRF(4)=0.500 | OBP(4)=0.417 | IBEP(2)=0.325 | BRF(50)=0.250
  Fold 3/10... ✅ MPRF(2)=0.556 | OBP(14)=0.544 | IBEP(4)=0.383 | BRF(50)=0.333
  Fold 4/10... ✅ MPRF(10)=0.617 | OBP(48)=0.608 | IBEP(1)=0.430 | BRF(50)=0.450
  Fold 5/10... ✅ MPRF(6)=0.493 | OBP(4)=0.487 | IBEP(2)=0.444 | BRF(50)=0.360
  Fold 6/10... ✅ MPRF(2)=0.578 | OBP(6)=0.517 | IBEP(3)=0.537 | BRF(50)=0.467
  Fold 7/10... ✅ MPRF(10)=0.495 | OBP(32)=0.538 | IBEP(2)=0.532 | BRF(50)=0.495
  Fold 8/10... ✅ MPRF(2)=0.475 | OBP(4)=0.554 | IBEP(1)=0.528 | BRF(50)=0.433
  Fold 9/10... ✅ MPRF(10)=0.496 | OBP(40)=0.493 | IBEP(1)=0.469 | BRF(50)=0.385
  Fold 10/10... ✅

# Mammography

In [ ]:
# ============================================================
# MPRF — Mammography Dataset
# Score combiné : α·F1_min + (1-α)·G-mean
# Métriques complètes + Tests statistiques
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score)
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ── Chargement fichier ARFF ───────────────────────────────────
lines = []
data_section = False
with open("/content/phpn1jVwe.arff", "r") as f:
    for line in f:
        line = line.strip()
        if line.lower() == "@data":
            data_section = True
            continue
        if data_section and line:
            lines.append(line)

rows = []
for line in lines:
    vals = line.split(",")
    features = [float(v) for v in vals[:6]]
    label = 1 if vals[6].strip().strip("'") == "1" else 0
    rows.append(features + [label])

df = pd.DataFrame(rows, columns=["attr1","attr2","attr3","attr4","attr5","attr6","target"])

TARGET = "target"
X = df[[c for c in df.columns if c != TARGET]].values
y = df[TARGET].values.astype(int)

counts = np.bincount(y)
print(f"Shape : {X.shape}  |  Classes : {counts}  |  IR ≈ {counts[0]/counts[1]:.1f}:1")

K_BIRF = 10

# ┌─────────────────────────────────────────────────────────┐
# │  ALPHA — paramètre du score combiné MPRF               │
# │  Score(S) = α·F1_min(S) + (1-α)·G-mean(S)             │
# │  α = 1.0  → pur F1_min                                 │
# │  α = 0.7  → recommandé                                 │
# │  α = 0.5  → équilibre F1_min et G-mean                 │
# └─────────────────────────────────────────────────────────┘
ALPHA = 0.6   # ← MODIFIER ICI

print(f"Score combiné : α={ALPHA:.1f} · F1_min + {1-ALPHA:.1f} · G-mean")

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)

    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

# ── Vote majoritaire ─────────────────────────────────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

# ── Score combiné pour la sélection ──────────────────────────
def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)

    return alpha * f1 + (1 - alpha) * gm

# ── MPRF — Forward Selection hill climbing ───────────────────
def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    """
    Hill climbing forward selection :
    1. Init : un arbre tiré aléatoirement (seed propre au fold)
    2. Critère : Score(S) = α·F1_min(S) + (1-α)·G-mean(S)
    3. À chaque étape : on évalue TOUS les candidats restants
       et on ajoute le meilleur même si égalité (plateau)
    4. best_score mis à jour UNIQUEMENT si amélioration stricte
    5. On s'arrête si tous les candidats dégradent ou si k atteint
    """
    n = len(trees)

    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)

    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)

    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected),
                                alpha)

    while len(selected) < k and remaining:

        best_cand       = None
        best_cand_score = -np.inf

        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score = score
                best_cand       = idx

        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break

    return selected

# ── OBP ──────────────────────────────────────────────────────
def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

# ── IBEP ─────────────────────────────────────────────────────
def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]

# ── Cliff's Delta ─────────────────────────────────────────────
def cliffs_delta(x, y):
    x, y = np.array(x), np.array(y)
    n, m = len(x), len(y)
    more = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    delta = (more - less) / (n * m)
    abs_d = abs(delta)
    if abs_d < 0.147:
        magnitude = "negligible"
    elif abs_d < 0.330:
        magnitude = "small"
    elif abs_d < 0.474:
        magnitude = "medium"
    else:
        magnitude = "large"
    return delta, magnitude

# ── display_results ──────────────────────────────────────────
def display_results(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    print(f"\n{'='*85}")
    print(f"  Résultats — {dataset_label}")
    print(f"{'='*85}")
    scores = {}
    for method in methods:
        vals = results.get(f"{method}_f1_min", [np.nan])
        scores[method] = np.mean(vals)
        size = np.mean(results.get(f"{method}_size",      [np.nan]))
        f1   = np.mean(results.get(f"{method}_f1_min",    [np.nan]))
        gm   = np.mean(results.get(f"{method}_gmean",     [np.nan]))
        aup  = np.mean(results.get(f"{method}_auprc",     [np.nan]))
        prec = np.mean(results.get(f"{method}_precision", [np.nan]))
        rec  = np.mean(results.get(f"{method}_recall",    [np.nan]))
        print(f"  {method:<6} | taille={size:5.1f} | "
              f"F1_min={f1:.3f} | G-mean={gm:.3f} | "
              f"AUPRC={aup:.3f} | Prec={prec:.3f} | Recall={rec:.3f}")
    winner = max(scores, key=scores.get)
    print(f"\n🏆 Gagnant (F1 minoritaire) : {winner} = {scores[winner]:.3f}")
    print(f"{'='*85}")

# ── Tests statistiques ────────────────────────────────────────
def statistical_analysis(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    metrics = ["f1_min", "gmean", "auprc"]
    ref     = "MPRF"

    print(f"\n{'='*75}")
    print(f"  Tests statistiques — {dataset_label}")
    print(f"{'='*75}")

    for metric in metrics:
        print(f"\n  Métrique : {metric.upper()}")
        print(f"  {'Comparaison':<20} {'p-value':>10} {'Significatif':>14} "
              f"{'Cliff δ':>10} {'Magnitude':>12}")
        print(f"  {'-'*66}")

        ref_vals = np.array(results.get(f"{ref}_{metric}", [np.nan]))

        for method in methods:
            if method == ref:
                continue
            comp_vals = np.array(results.get(f"{method}_{metric}", [np.nan]))
            try:
                if np.all(ref_vals == comp_vals):
                    p_val = 1.0
                else:
                    _, p_val = wilcoxon(ref_vals, comp_vals)
            except Exception:
                p_val = np.nan

            sig   = "✅ Oui" if (not np.isnan(p_val) and p_val < 0.05) else "❌ Non"
            delta, magnitude = cliffs_delta(ref_vals, comp_vals)

            print(f"  {ref} vs {method:<14} {p_val:>10.4f} {sig:>14} "
                  f"{delta:>10.3f} {magnitude:>12}")

    print(f"\n  Seuil α = 0.05 | "
          f"Cliff δ : negligible<0.147, small<0.330, medium<0.474, large≥0.474")
    print(f"{'='*75}")

# ── run_experiment_v2 ─────────────────────────────────────────
def run_experiment_v2(X, y, dataset_name, n_trees=50, k=K_BIRF, n_splits=10):

    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = defaultdict(list)

    print(f"\n{'='*60}")
    print(f"Dataset : {dataset_name}")
    print(f"RF balanced : {n_trees} arbres | MPRF : k={k} | Folds : {n_splits}")
    print(f"Score : α={ALPHA:.1f}·F1_min + {1-ALPHA:.1f}·G-mean")
    print(f"OBP et IBEP : taille automatique")
    print(f"{'='*60}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)

        X_val       = X_train[val_idx]
        X_train_sub = X_train[train_sub_idx]
        y_val       = y_train[val_idx]
        y_train_sub = y_train[train_sub_idx]

        rf = RandomForestClassifier(
            n_estimators=n_trees,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        # BRF
        y_pred_brf  = rf.predict(X_test)
        y_proba_brf = rf.predict_proba(X_test)[:, 1]
        for k_m, v in compute_metrics(y_test, y_pred_brf, y_proba_brf).items():
            results[f'BRF_{k_m}'].append(v)
        results['BRF_size'].append(n_trees)

        # MPRF
        selected_mprf = MPRF(trees, X_val, y_val, k, fold, ALPHA)
        y_pred_mprf   = ensemble_predict(trees, X_test, selected_mprf)
        y_proba_mprf  = ensemble_proba(trees, X_test, selected_mprf)
        for k_m, v in compute_metrics(y_test, y_pred_mprf, y_proba_mprf).items():
            results[f'MPRF_{k_m}'].append(v)
        results['MPRF_size'].append(len(selected_mprf))

        # OBP
        selected_obp  = OBP(trees, X_val, y_val, n_trees)
        y_pred_obp    = ensemble_predict(trees, X_test, selected_obp)
        y_proba_obp   = ensemble_proba(trees, X_test, selected_obp)
        for k_m, v in compute_metrics(y_test, y_pred_obp, y_proba_obp).items():
            results[f'OBP_{k_m}'].append(v)
        results['OBP_size'].append(len(selected_obp))

        # IBEP
        selected_ibep = IBEP(trees, X_val, y_val, n_trees)
        y_pred_ibep   = ensemble_predict(trees, X_test, selected_ibep)
        y_proba_ibep  = ensemble_proba(trees, X_test, selected_ibep)
        for k_m, v in compute_metrics(y_test, y_pred_ibep, y_proba_ibep).items():
            results[f'IBEP_{k_m}'].append(v)
        results['IBEP_size'].append(len(selected_ibep))

        print(f"✅ MPRF({len(selected_mprf)})={np.mean(results['MPRF_f1_min']):.3f} | "
              f"OBP({len(selected_obp)})={np.mean(results['OBP_f1_min']):.3f} | "
              f"IBEP({len(selected_ibep)})={np.mean(results['IBEP_f1_min']):.3f} | "
              f"BRF({n_trees})={np.mean(results['BRF_f1_min']):.3f}")

    return results

# ── Lancement ────────────────────────────────────────────────
results_mammo = run_experiment_v2(
    X, y, "Mammography (microcalcifications)",
    n_trees=50, k=K_BIRF, n_splits=10
)

display_results(results_mammo,
                "Mammography — microcalcification=1 (minoritaire)")
statistical_analysis(results_mammo,
                     "Mammography — microcalcification=1 (minoritaire)")

# ── Tableau fold par fold ─────────────────────────────────────
print(f"\n{'='*60}")
print("  Valeurs fold par fold — F1_min")
print(f"{'='*60}")
methods_list = ["BRF", "MPRF", "OBP", "IBEP"]
print(f"{'Fold':<6} " + " ".join(f"{m:>8}" for m in methods_list))
print("-" * 42)
for i in range(10):
    row = f"{i+1:<6} "
    for m in methods_list:
        val = results_mammo[f"{m}_f1_min"][i]
        row += f"  {val:.3f}  "
    print(row)
print("-" * 42)
print(f"{'Moy':<6} ", end="")
for m in methods_list:
    print(f"  {np.mean(results_mammo[f'{m}_f1_min']):.3f}  ", end="")
print()
print(f"{'Std':<6} ", end="")
for m in methods_list:
    print(f"  {np.std(results_mammo[f'{m}_f1_min']):.3f}  ", end="")
print()

Shape : (11183, 6)  |  Classes : [10923   260]  |  IR ≈ 42.0:1
Score combiné : α=0.6 · F1_min + 0.4 · G-mean

Dataset : Mammography (microcalcifications)
RF balanced : 50 arbres | MPRF : k=10 | Folds : 10
Score : α=0.6·F1_min + 0.4·G-mean
OBP et IBEP : taille automatique
  Fold 1/10... ✅ MPRF(4)=0.680 | OBP(1)=0.627 | IBEP(8)=0.756 | BRF(50)=0.650
  Fold 2/10... ✅ MPRF(2)=0.640 | OBP(4)=0.621 | IBEP(9)=0.711 | BRF(50)=0.695
  Fold 3/10... ✅ MPRF(4)=0.659 | OBP(4)=0.654 | IBEP(6)=0.696 | BRF(50)=0.669
  Fold 4/10... ✅ MPRF(4)=0.646 | OBP(6)=0.634 | IBEP(10)=0.673 | BRF(50)=0.656
  Fold 5/10... ✅ MPRF(4)=0.645 | OBP(1)=0.627 | IBEP(9)=0.658 | BRF(50)=0.643
  Fold 6/10... ✅ MPRF(4)=0.641 | OBP(7)=0.622 | IBEP(12)=0.638 | BRF(50)=0.621
  Fold 7/10... ✅ MPRF(4)=0.616 | OBP(4)=0.591 | IBEP(8)=0.622 | BRF(50)=0.612
  Fold 8/10... ✅ MPRF(4)=0.609 | OBP(1)=0.588 | IBEP(7)=0.631 | BRF(50)=0.613
  Fold 9/10... ✅ MPRF(2)=0.584 | OBP(8)=0.581 | IBEP(10)=0.613 | BRF(50)=0.602
  Fold 10/10... ✅ MPRF(

# Diabetes

In [ ]:
# ============================================================
# MPRF — Pima Indians Diabetes Dataset
# Score combiné : α·F1_min + (1-α)·G-mean
# Métriques complètes + Tests statistiques
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score)
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
df = pd.read_csv("3. diabetes.csv")
df.fillna(df.median(numeric_only=True), inplace=True)

TARGET = "Outcome"
X = df[[c for c in df.columns if c != TARGET]].values
y = df[TARGET].values.astype(int)

counts = np.bincount(y)
print(f"Shape : {X.shape}  |  Classes : {counts}  |  IR ≈ {counts[0]/counts[1]:.1f}:1")

K_BIRF = 10

# ┌─────────────────────────────────────────────────────────┐
# │  ALPHA — paramètre du score combiné MPRF               │
# │  Score(S) = α·F1_min(S) + (1-α)·G-mean(S)             │
# │  α = 1.0  → pur F1_min                                 │
# │  α = 0.7  → recommandé                                 │
# │  α = 0.5  → équilibre F1_min et G-mean                 │
# └─────────────────────────────────────────────────────────┘
ALPHA = 0.6   # ← MODIFIER ICI

print(f"Score combiné : α={ALPHA:.1f} · F1_min + {1-ALPHA:.1f} · G-mean")

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)

    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

# ── Vote majoritaire ─────────────────────────────────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

# ── Score combiné pour la sélection ──────────────────────────
def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)

    return alpha * f1 + (1 - alpha) * gm

# ── MPRF — Forward Selection hill climbing ───────────────────
def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    """
    Hill climbing forward selection :
    1. Init : un arbre tiré aléatoirement (seed propre au fold)
    2. Critère : Score(S) = α·F1_min(S) + (1-α)·G-mean(S)
    3. À chaque étape : on évalue TOUS les candidats restants
       et on ajoute le meilleur même si égalité (plateau)
    4. best_score mis à jour UNIQUEMENT si amélioration stricte
    5. On s'arrête si tous les candidats dégradent ou si k atteint
    """
    n = len(trees)

    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)

    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)

    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected),
                                alpha)

    while len(selected) < k and remaining:

        best_cand       = None
        best_cand_score = -np.inf

        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score = score
                best_cand       = idx

        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break

    return selected

# ── OBP ──────────────────────────────────────────────────────
def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

# ── IBEP ─────────────────────────────────────────────────────
def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]

# ── Cliff's Delta ─────────────────────────────────────────────
def cliffs_delta(x, y):
    x, y = np.array(x), np.array(y)
    n, m = len(x), len(y)
    more = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    delta = (more - less) / (n * m)
    abs_d = abs(delta)
    if abs_d < 0.147:
        magnitude = "negligible"
    elif abs_d < 0.330:
        magnitude = "small"
    elif abs_d < 0.474:
        magnitude = "medium"
    else:
        magnitude = "large"
    return delta, magnitude

# ── display_results ──────────────────────────────────────────
def display_results(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    print(f"\n{'='*85}")
    print(f"  Résultats — {dataset_label}")
    print(f"{'='*85}")
    scores = {}
    for method in methods:
        vals = results.get(f"{method}_f1_min", [np.nan])
        scores[method] = np.mean(vals)
        size = np.mean(results.get(f"{method}_size",      [np.nan]))
        f1   = np.mean(results.get(f"{method}_f1_min",    [np.nan]))
        gm   = np.mean(results.get(f"{method}_gmean",     [np.nan]))
        aup  = np.mean(results.get(f"{method}_auprc",     [np.nan]))
        prec = np.mean(results.get(f"{method}_precision", [np.nan]))
        rec  = np.mean(results.get(f"{method}_recall",    [np.nan]))
        print(f"  {method:<6} | taille={size:5.1f} | "
              f"F1_min={f1:.3f} | G-mean={gm:.3f} | "
              f"AUPRC={aup:.3f} | Prec={prec:.3f} | Recall={rec:.3f}")
    winner = max(scores, key=scores.get)
    print(f"\n🏆 Gagnant (F1 minoritaire) : {winner} = {scores[winner]:.3f}")
    print(f"{'='*85}")

# ── Tests statistiques ────────────────────────────────────────
def statistical_analysis(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    metrics = ["f1_min", "gmean", "auprc"]
    ref     = "MPRF"

    print(f"\n{'='*75}")
    print(f"  Tests statistiques — {dataset_label}")
    print(f"{'='*75}")

    for metric in metrics:
        print(f"\n  Métrique : {metric.upper()}")
        print(f"  {'Comparaison':<20} {'p-value':>10} {'Significatif':>14} "
              f"{'Cliff δ':>10} {'Magnitude':>12}")
        print(f"  {'-'*66}")

        ref_vals = np.array(results.get(f"{ref}_{metric}", [np.nan]))

        for method in methods:
            if method == ref:
                continue
            comp_vals = np.array(results.get(f"{method}_{metric}", [np.nan]))
            try:
                if np.all(ref_vals == comp_vals):
                    p_val = 1.0
                else:
                    _, p_val = wilcoxon(ref_vals, comp_vals)
            except Exception:
                p_val = np.nan

            sig   = "✅ Oui" if (not np.isnan(p_val) and p_val < 0.05) else "❌ Non"
            delta, magnitude = cliffs_delta(ref_vals, comp_vals)

            print(f"  {ref} vs {method:<14} {p_val:>10.4f} {sig:>14} "
                  f"{delta:>10.3f} {magnitude:>12}")

    print(f"\n  Seuil α = 0.05 | "
          f"Cliff δ : negligible<0.147, small<0.330, medium<0.474, large≥0.474")
    print(f"{'='*75}")

# ── run_experiment_v2 ─────────────────────────────────────────
def run_experiment_v2(X, y, dataset_name, n_trees=50, k=K_BIRF, n_splits=10):

    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = defaultdict(list)

    print(f"\n{'='*60}")
    print(f"Dataset : {dataset_name}")
    print(f"RF balanced : {n_trees} arbres | MPRF : k={k} | Folds : {n_splits}")
    print(f"Score : α={ALPHA:.1f}·F1_min + {1-ALPHA:.1f}·G-mean")
    print(f"OBP et IBEP : taille automatique")
    print(f"{'='*60}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)

        X_val       = X_train[val_idx]
        X_train_sub = X_train[train_sub_idx]
        y_val       = y_train[val_idx]
        y_train_sub = y_train[train_sub_idx]

        rf = RandomForestClassifier(
            n_estimators=n_trees,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        # BRF
        y_pred_brf  = rf.predict(X_test)
        y_proba_brf = rf.predict_proba(X_test)[:, 1]
        for k_m, v in compute_metrics(y_test, y_pred_brf, y_proba_brf).items():
            results[f'BRF_{k_m}'].append(v)
        results['BRF_size'].append(n_trees)

        # MPRF
        selected_mprf = MPRF(trees, X_val, y_val, k, fold, ALPHA)
        y_pred_mprf   = ensemble_predict(trees, X_test, selected_mprf)
        y_proba_mprf  = ensemble_proba(trees, X_test, selected_mprf)
        for k_m, v in compute_metrics(y_test, y_pred_mprf, y_proba_mprf).items():
            results[f'MPRF_{k_m}'].append(v)
        results['MPRF_size'].append(len(selected_mprf))

        # OBP
        selected_obp  = OBP(trees, X_val, y_val, n_trees)
        y_pred_obp    = ensemble_predict(trees, X_test, selected_obp)
        y_proba_obp   = ensemble_proba(trees, X_test, selected_obp)
        for k_m, v in compute_metrics(y_test, y_pred_obp, y_proba_obp).items():
            results[f'OBP_{k_m}'].append(v)
        results['OBP_size'].append(len(selected_obp))

        # IBEP
        selected_ibep = IBEP(trees, X_val, y_val, n_trees)
        y_pred_ibep   = ensemble_predict(trees, X_test, selected_ibep)
        y_proba_ibep  = ensemble_proba(trees, X_test, selected_ibep)
        for k_m, v in compute_metrics(y_test, y_pred_ibep, y_proba_ibep).items():
            results[f'IBEP_{k_m}'].append(v)
        results['IBEP_size'].append(len(selected_ibep))

        print(f"✅ MPRF({len(selected_mprf)})={np.mean(results['MPRF_f1_min']):.3f} | "
              f"OBP({len(selected_obp)})={np.mean(results['OBP_f1_min']):.3f} | "
              f"IBEP({len(selected_ibep)})={np.mean(results['IBEP_f1_min']):.3f} | "
              f"BRF({n_trees})={np.mean(results['BRF_f1_min']):.3f}")

    return results

# ── Lancement ────────────────────────────────────────────────
results_diab = run_experiment_v2(
    X, y, "Pima Indians Diabetes",
    n_trees=50, k=K_BIRF, n_splits=10
)

display_results(results_diab,
                "Pima Indians Diabetes — diabétique=1 (minoritaire)")
statistical_analysis(results_diab,
                     "Pima Indians Diabetes — diabétique=1 (minoritaire)")

# ── Tableau fold par fold ─────────────────────────────────────
print(f"\n{'='*60}")
print("  Valeurs fold par fold — F1_min")
print(f"{'='*60}")
methods_list = ["BRF", "MPRF", "OBP", "IBEP"]
print(f"{'Fold':<6} " + " ".join(f"{m:>8}" for m in methods_list))
print("-" * 42)
for i in range(10):
    row = f"{i+1:<6} "
    for m in methods_list:
        val = results_diab[f"{m}_f1_min"][i]
        row += f"  {val:.3f}  "
    print(row)
print("-" * 42)
print(f"{'Moy':<6} ", end="")
for m in methods_list:
    print(f"  {np.mean(results_diab[f'{m}_f1_min']):.3f}  ", end="")
print()
print(f"{'Std':<6} ", end="")
for m in methods_list:
    print(f"  {np.std(results_diab[f'{m}_f1_min']):.3f}  ", end="")
print()

Shape : (768, 8)  |  Classes : [500 268]  |  IR ≈ 1.9:1
Score combiné : α=0.6 · F1_min + 0.4 · G-mean

Dataset : Pima Indians Diabetes
RF balanced : 50 arbres | MPRF : k=10 | Folds : 10
Score : α=0.6·F1_min + 0.4·G-mean
OBP et IBEP : taille automatique
  Fold 1/10... ✅ MPRF(4)=0.738 | OBP(7)=0.643 | IBEP(11)=0.582 | BRF(50)=0.654
  Fold 2/10... ✅ MPRF(3)=0.658 | OBP(7)=0.619 | IBEP(13)=0.591 | BRF(50)=0.638
  Fold 3/10... ✅ MPRF(10)=0.649 | OBP(7)=0.595 | IBEP(10)=0.588 | BRF(50)=0.618
  Fold 4/10... ✅ MPRF(4)=0.661 | OBP(11)=0.623 | IBEP(12)=0.632 | BRF(50)=0.646
  Fold 5/10... ✅ MPRF(2)=0.634 | OBP(2)=0.617 | IBEP(7)=0.594 | BRF(50)=0.612
  Fold 6/10... ✅ MPRF(2)=0.635 | OBP(6)=0.627 | IBEP(9)=0.600 | BRF(50)=0.608
  Fold 7/10... ✅ MPRF(4)=0.632 | OBP(10)=0.626 | IBEP(12)=0.596 | BRF(50)=0.595
  Fold 8/10... ✅ MPRF(10)=0.639 | OBP(4)=0.625 | IBEP(8)=0.600 | BRF(50)=0.598
  Fold 9/10... ✅ MPRF(4)=0.634 | OBP(8)=0.626 | IBEP(9)=0.595 | BRF(50)=0.599
  Fold 10/10... ✅ MPRF(4)=0.619 | OB

# sick thyroid

In [ ]:
# ============================================================
# MPRF — Thyroid Disease (sick) Dataset
# Score combiné : α·F1_min + (1-α)·G-mean
# Métriques complètes + Tests statistiques
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
# Colonnes du fichier sick.data (UCI Thyroid)
col_names = [
    "age", "sex", "on_thyroxine", "query_on_thyroxine",
    "on_antithyroid_medication", "sick", "pregnant", "thyroid_surgery",
    "I131_treatment", "query_hypothyroid", "query_hyperthyroid",
    "lithium", "goitre", "tumor", "hypopituitary", "psych",
    "TSH_measured", "TSH", "T3_measured", "T3",
    "TT4_measured", "TT4", "T4U_measured", "T4U",
    "FTI_measured", "FTI", "TBG_measured", "TBG",
    "referral_source", "target"
]

df = pd.read_csv("Thyroid_sick.arff", header=None,
                 names=col_names, na_values="?")

# Cible : sick vs negative
df["target"] = df["target"].astype(str).str.strip()
df["target"] = df["target"].map(lambda x:
    1 if x.startswith("sick") else
    (0 if x.startswith("negative") else np.nan)
)
df = df.dropna(subset=["target"])
df["target"] = df["target"].astype(int)

print("Distribution cible :", df["target"].value_counts().to_dict())

# Features
TARGET = "target"
drop_cols = [TARGET, "referral_source"]
feature_cols = [c for c in df.columns if c not in drop_cols]
df_feat = df[feature_cols].copy()

for col in df_feat.columns:
    if df_feat[col].dtype == object:
        df_feat[col] = df_feat[col].map(
            lambda x: 1 if str(x) == "t" else
                      (0 if str(x) == "f" else
                      (1 if str(x) == "M" else
                      (0 if str(x) == "F" else np.nan)))
        )
        df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")

df_feat = df_feat.apply(pd.to_numeric, errors="coerce")
df_feat.fillna(df_feat.median(numeric_only=True), inplace=True)

X = df_feat.values
y = df[TARGET].values.astype(int)

counts = np.bincount(y)
print(f"Shape : {X.shape}  |  Classes : {counts}  |  IR ≈ {counts[0]/counts[1]:.1f}:1")

K_BIRF = 10

# ┌─────────────────────────────────────────────────────────┐
# │  ALPHA — Score(S) = α·F1_min(S) + (1-α)·G-mean(S)     │
# └─────────────────────────────────────────────────────────┘
ALPHA = 0   # ← MODIFIER ICI

print(f"Score combiné : α={ALPHA:.1f} · F1_min + {1-ALPHA:.1f} · G-mean")

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)
    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)
    return alpha * f1 + (1 - alpha) * gm

def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    n = len(trees)
    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)
    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)
    best_score = combined_score(y_val, ensemble_predict(trees, X_val, selected), alpha)
    while len(selected) < k and remaining:
        best_cand, best_cand_score = None, -np.inf
        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score, best_cand = score, idx
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break
    return selected

def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]

def cliffs_delta(x, y):
    x, y = np.array(x), np.array(y)
    n, m = len(x), len(y)
    more = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    delta = (more - less) / (n * m)
    abs_d = abs(delta)
    if abs_d < 0.147:   magnitude = "negligible"
    elif abs_d < 0.330: magnitude = "small"
    elif abs_d < 0.474: magnitude = "medium"
    else:               magnitude = "large"
    return delta, magnitude

def display_results(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    print(f"\n{'='*85}")
    print(f"  Résultats — {dataset_label}")
    print(f"{'='*85}")
    scores = {}
    for method in methods:
        scores[method] = np.mean(results.get(f"{method}_f1_min", [np.nan]))
        size = np.mean(results.get(f"{method}_size",      [np.nan]))
        f1   = np.mean(results.get(f"{method}_f1_min",    [np.nan]))
        gm   = np.mean(results.get(f"{method}_gmean",     [np.nan]))
        aup  = np.mean(results.get(f"{method}_auprc",     [np.nan]))
        prec = np.mean(results.get(f"{method}_precision", [np.nan]))
        rec  = np.mean(results.get(f"{method}_recall",    [np.nan]))
        print(f"  {method:<6} | taille={size:5.1f} | F1_min={f1:.3f} | "
              f"G-mean={gm:.3f} | AUPRC={aup:.3f} | Prec={prec:.3f} | Recall={rec:.3f}")
    winner = max(scores, key=scores.get)
    print(f"\n🏆 Gagnant : {winner} = {scores[winner]:.3f}")
    print(f"{'='*85}")

def statistical_analysis(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    metrics = ["f1_min", "gmean", "auprc"]
    ref     = "MPRF"
    print(f"\n{'='*75}")
    print(f"  Tests statistiques — {dataset_label}")
    print(f"{'='*75}")
    for metric in metrics:
        print(f"\n  Métrique : {metric.upper()}")
        print(f"  {'Comparaison':<20} {'p-value':>10} {'Significatif':>14} "
              f"{'Cliff δ':>10} {'Magnitude':>12}")
        print(f"  {'-'*66}")
        ref_vals = np.array(results.get(f"{ref}_{metric}", [np.nan]))
        for method in methods:
            if method == ref:
                continue
            comp_vals = np.array(results.get(f"{method}_{metric}", [np.nan]))
            try:
                _, p_val = wilcoxon(ref_vals, comp_vals) if not np.all(ref_vals == comp_vals) else (None, 1.0)
            except Exception:
                p_val = np.nan
            sig = "✅ Oui" if (not np.isnan(p_val) and p_val < 0.05) else "❌ Non"
            delta, magnitude = cliffs_delta(ref_vals, comp_vals)
            print(f"  {ref} vs {method:<14} {p_val:>10.4f} {sig:>14} "
                  f"{delta:>10.3f} {magnitude:>12}")
    print(f"\n  Seuil α=0.05 | Cliff δ: negligible<0.147, small<0.330, medium<0.474, large≥0.474")
    print(f"{'='*75}")

def run_experiment_v2(X, y, dataset_name, n_trees=50, k=K_BIRF, n_splits=10):
    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = defaultdict(list)
    print(f"\n{'='*60}")
    print(f"Dataset : {dataset_name}")
    print(f"RF balanced : {n_trees} arbres | MPRF : k={k} | Folds : {n_splits}")
    print(f"Score : α={ALPHA:.1f}·F1_min + {1-ALPHA:.1f}·G-mean")
    print(f"{'='*60}")
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)
        X_val, y_val           = X_train[val_idx], y_train[val_idx]
        X_train_sub, y_train_sub = X_train[train_sub_idx], y_train[train_sub_idx]
        rf = RandomForestClassifier(n_estimators=n_trees, class_weight='balanced',
                                    random_state=42, n_jobs=-1)
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_
        for name, sel, pred_fn in [
            ("BRF",  None, lambda: (rf.predict(X_test), rf.predict_proba(X_test)[:, 1])),
        ]:
            y_pred, y_proba = rf.predict(X_test), rf.predict_proba(X_test)[:, 1]
            for k_m, v in compute_metrics(y_test, y_pred, y_proba).items():
                results[f'BRF_{k_m}'].append(v)
            results['BRF_size'].append(n_trees)

        for name, func in [("MPRF", lambda: MPRF(trees, X_val, y_val, k, fold, ALPHA)),
                            ("OBP",  lambda: OBP(trees, X_val, y_val, n_trees)),
                            ("IBEP", lambda: IBEP(trees, X_val, y_val, n_trees))]:
            sel   = func()
            y_pred  = ensemble_predict(trees, X_test, sel)
            y_proba = ensemble_proba(trees, X_test, sel)
            for k_m, v in compute_metrics(y_test, y_pred, y_proba).items():
                results[f'{name}_{k_m}'].append(v)
            results[f'{name}_size'].append(len(sel))

        print(f"✅ MPRF({len(MPRF(trees,X_val,y_val,k,fold,ALPHA))})="
              f"{np.mean(results['MPRF_f1_min']):.3f} | "
              f"OBP={np.mean(results['OBP_f1_min']):.3f} | "
              f"IBEP={np.mean(results['IBEP_f1_min']):.3f} | "
              f"BRF={np.mean(results['BRF_f1_min']):.3f}")
    return results

results_thyroid = run_experiment_v2(X, y, "Thyroid Disease (sick)")
display_results(results_thyroid, "Thyroid Disease — sick=1 (minoritaire)")
statistical_analysis(results_thyroid, "Thyroid Disease — sick=1 (minoritaire)")

methods_list = ["BRF", "MPRF", "OBP", "IBEP"]
print(f"\n{'Fold':<6} " + " ".join(f"{m:>8}" for m in methods_list))
print("-" * 42)
for i in range(10):
    row = f"{i+1:<6} "
    for m in methods_list:
        row += f"  {results_thyroid[f'{m}_f1_min'][i]:.3f}  "
    print(row)
print("-" * 42)
print(f"{'Moy':<6} ", end="")
for m in methods_list:
    print(f"  {np.mean(results_thyroid[f'{m}_f1_min']):.3f}  ", end="")
print()
print(f"{'Std':<6} ", end="")
for m in methods_list:
    print(f"  {np.std(results_thyroid[f'{m}_f1_min']):.3f}  ", end="")
print()

Distribution cible : {0: 3541, 1: 231}
Shape : (3772, 28)  |  Classes : [3541  231]  |  IR ≈ 15.3:1
Score combiné : α=0.0 · F1_min + 1.0 · G-mean

Dataset : Thyroid Disease (sick)
RF balanced : 50 arbres | MPRF : k=10 | Folds : 10
Score : α=0.0·F1_min + 1.0·G-mean
  Fold 1/10... ✅ MPRF(4)=0.863 | OBP=0.792 | IBEP=0.894 | BRF=0.894
  Fold 2/10... ✅ MPRF(2)=0.863 | OBP=0.796 | IBEP=0.852 | BRF=0.837
  Fold 3/10... ✅ MPRF(2)=0.855 | OBP=0.809 | IBEP=0.842 | BRF=0.850
  Fold 4/10... ✅ MPRF(2)=0.849 | OBP=0.818 | IBEP=0.836 | BRF=0.850
  Fold 5/10... ✅ MPRF(2)=0.841 | OBP=0.821 | IBEP=0.841 | BRF=0.861
  Fold 6/10... ✅ MPRF(2)=0.841 | OBP=0.831 | IBEP=0.834 | BRF=0.848
  Fold 7/10... ✅ MPRF(2)=0.857 | OBP=0.848 | IBEP=0.849 | BRF=0.863
  Fold 8/10... ✅ MPRF(2)=0.843 | OBP=0.846 | IBEP=0.844 | BRF=0.852
  Fold 9/10... ✅ MPRF(2)=0.836 | OBP=0.846 | IBEP=0.843 | BRF=0.848
  Fold 10/10... ✅ MPRF(2)=0.839 | OBP=0.854 | IBEP=0.848 | BRF=0.856

  Résultats — Thyroid Disease — sick=1 (minoritaire)


# ecoli

In [ ]:
# ============================================================
# MPRF — Ecoli Dataset (imbalanced binary)
# Score combiné : α·F1_min + (1-α)·G-mean
# Métriques complètes + Tests statistiques
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              average_precision_score)
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
col_names = ["seq_name", "mcg", "gvh", "lip", "chg", "aac", "alm1", "alm2", "class"]

df = pd.read_csv("/content/ecoli.data", header=None,
                 names=col_names, sep=r'\s+')

print("Classes originales :", df["class"].value_counts().to_dict())

# ── Cible binaire : im (minoritaire=1) vs tout le reste (0) ──
# im est la classe la plus rare — IR ≈ 34:1
# On peut aussi choisir pp, cp selon l'IR voulu
TARGET_CLASS = "imU"   # ← MODIFIER ICI si besoin (im, imU, imL, imS, pp, cp, om, omL)

df["target"] = df["class"].map(lambda x: 1 if x == TARGET_CLASS else 0)
df = df.drop(columns=["seq_name", "class"])

print(f"Cible : {TARGET_CLASS} vs reste")
print("Distribution :", df["target"].value_counts().to_dict())

TARGET = "target"
X = df[[c for c in df.columns if c != TARGET]].values
y = df[TARGET].values.astype(int)

counts = np.bincount(y)
print(f"Shape : {X.shape}  |  Classes : {counts}  |  IR ≈ {counts[0]/counts[1]:.1f}:1")

K_BIRF = 10

# ┌─────────────────────────────────────────────────────────┐
# │  ALPHA — Score(S) = α·F1_min(S) + (1-α)·G-mean(S)     │
# │  α = 0.7 → recommandé                                  │
# └─────────────────────────────────────────────────────────┘
ALPHA = 0.7   # ← MODIFIER ICI

print(f"Score combiné : α={ALPHA:.1f} · F1_min + {1-ALPHA:.1f} · G-mean")

# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)
    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

# ── Vote majoritaire ─────────────────────────────────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

# ── Score combiné ─────────────────────────────────────────────
def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)
    return alpha * f1 + (1 - alpha) * gm

# ── MPRF — Forward Selection hill climbing ───────────────────
def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    """
    Hill climbing forward selection :
    1. Init : un arbre tiré aléatoirement (seed propre au fold)
    2. Critère : Score(S) = α·F1_min(S) + (1-α)·G-mean(S)
    3. Avancer si meilleur OU égal (plateau accepté)
    4. best_score mis à jour UNIQUEMENT si amélioration stricte
    5. Arrêt si dégradation ou k atteint
    """
    n = len(trees)
    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)
    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)
    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected),
                                alpha)
    while len(selected) < k and remaining:
        best_cand, best_cand_score = None, -np.inf
        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score, best_cand = score, idx
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break
    return selected

# ── OBP ──────────────────────────────────────────────────────
def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

# ── IBEP ─────────────────────────────────────────────────────
def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]

# ── Cliff's Delta ─────────────────────────────────────────────
def cliffs_delta(x, y):
    x, y = np.array(x), np.array(y)
    n, m = len(x), len(y)
    more = sum(1 for xi in x for yj in y if xi > yj)
    less = sum(1 for xi in x for yj in y if xi < yj)
    delta = (more - less) / (n * m)
    abs_d = abs(delta)
    if abs_d < 0.147:   magnitude = "negligible"
    elif abs_d < 0.330: magnitude = "small"
    elif abs_d < 0.474: magnitude = "medium"
    else:               magnitude = "large"
    return delta, magnitude

# ── display_results ──────────────────────────────────────────
def display_results(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    print(f"\n{'='*85}")
    print(f"  Résultats — {dataset_label}")
    print(f"{'='*85}")
    scores = {}
    for method in methods:
        scores[method] = np.mean(results.get(f"{method}_f1_min", [np.nan]))
        size = np.mean(results.get(f"{method}_size",      [np.nan]))
        f1   = np.mean(results.get(f"{method}_f1_min",    [np.nan]))
        gm   = np.mean(results.get(f"{method}_gmean",     [np.nan]))
        aup  = np.mean(results.get(f"{method}_auprc",     [np.nan]))
        prec = np.mean(results.get(f"{method}_precision", [np.nan]))
        rec  = np.mean(results.get(f"{method}_recall",    [np.nan]))
        print(f"  {method:<6} | taille={size:5.1f} | F1_min={f1:.3f} | "
              f"G-mean={gm:.3f} | AUPRC={aup:.3f} | Prec={prec:.3f} | Recall={rec:.3f}")
    winner = max(scores, key=scores.get)
    print(f"\n🏆 Gagnant : {winner} = {scores[winner]:.3f}")
    print(f"{'='*85}")

# ── Tests statistiques ────────────────────────────────────────
def statistical_analysis(results, dataset_label):
    methods = ["BRF", "MPRF", "OBP", "IBEP"]
    metrics = ["f1_min", "gmean", "auprc"]
    ref     = "MPRF"
    print(f"\n{'='*75}")
    print(f"  Tests statistiques — {dataset_label}")
    print(f"{'='*75}")
    for metric in metrics:
        print(f"\n  Métrique : {metric.upper()}")
        print(f"  {'Comparaison':<20} {'p-value':>10} {'Significatif':>14} "
              f"{'Cliff δ':>10} {'Magnitude':>12}")
        print(f"  {'-'*66}")
        ref_vals = np.array(results.get(f"{ref}_{metric}", [np.nan]))
        for method in methods:
            if method == ref:
                continue
            comp_vals = np.array(results.get(f"{method}_{metric}", [np.nan]))
            try:
                _, p_val = wilcoxon(ref_vals, comp_vals) \
                    if not np.all(ref_vals == comp_vals) else (None, 1.0)
            except Exception:
                p_val = np.nan
            sig = "✅ Oui" if (not np.isnan(p_val) and p_val < 0.05) else "❌ Non"
            delta, magnitude = cliffs_delta(ref_vals, comp_vals)
            print(f"  {ref} vs {method:<14} {p_val:>10.4f} {sig:>14} "
                  f"{delta:>10.3f} {magnitude:>12}")
    print(f"\n  Seuil α=0.05 | Cliff δ: negligible<0.147, small<0.330, "
          f"medium<0.474, large≥0.474")
    print(f"{'='*75}")

# ── run_experiment_v2 ─────────────────────────────────────────
def run_experiment_v2(X, y, dataset_name, n_trees=50, k=K_BIRF, n_splits=10):
    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = defaultdict(list)
    print(f"\n{'='*60}")
    print(f"Dataset : {dataset_name}")
    print(f"RF balanced : {n_trees} arbres | MPRF : k={k} | Folds : {n_splits}")
    print(f"Score : α={ALPHA:.1f}·F1_min + {1-ALPHA:.1f}·G-mean")
    print(f"{'='*60}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/{n_splits}...", end=" ")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)
        X_val, y_val             = X_train[val_idx], y_train[val_idx]
        X_train_sub, y_train_sub = X_train[train_sub_idx], y_train[train_sub_idx]

        rf = RandomForestClassifier(n_estimators=n_trees, class_weight='balanced',
                                    random_state=42, n_jobs=-1)
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        # BRF
        y_pred_brf  = rf.predict(X_test)
        y_proba_brf = rf.predict_proba(X_test)[:, 1]
        for k_m, v in compute_metrics(y_test, y_pred_brf, y_proba_brf).items():
            results[f'BRF_{k_m}'].append(v)
        results['BRF_size'].append(n_trees)

        # MPRF
        selected_mprf = MPRF(trees, X_val, y_val, k, fold, ALPHA)
        y_pred_mprf   = ensemble_predict(trees, X_test, selected_mprf)
        y_proba_mprf  = ensemble_proba(trees, X_test, selected_mprf)
        for k_m, v in compute_metrics(y_test, y_pred_mprf, y_proba_mprf).items():
            results[f'MPRF_{k_m}'].append(v)
        results['MPRF_size'].append(len(selected_mprf))

        # OBP
        selected_obp  = OBP(trees, X_val, y_val, n_trees)
        y_pred_obp    = ensemble_predict(trees, X_test, selected_obp)
        y_proba_obp   = ensemble_proba(trees, X_test, selected_obp)
        for k_m, v in compute_metrics(y_test, y_pred_obp, y_proba_obp).items():
            results[f'OBP_{k_m}'].append(v)
        results['OBP_size'].append(len(selected_obp))

        # IBEP
        selected_ibep = IBEP(trees, X_val, y_val, n_trees)
        y_pred_ibep   = ensemble_predict(trees, X_test, selected_ibep)
        y_proba_ibep  = ensemble_proba(trees, X_test, selected_ibep)
        for k_m, v in compute_metrics(y_test, y_pred_ibep, y_proba_ibep).items():
            results[f'IBEP_{k_m}'].append(v)
        results['IBEP_size'].append(len(selected_ibep))

        print(f"✅ MPRF({len(selected_mprf)})={np.mean(results['MPRF_f1_min']):.3f} | "
              f"OBP({len(selected_obp)})={np.mean(results['OBP_f1_min']):.3f} | "
              f"IBEP({len(selected_ibep)})={np.mean(results['IBEP_f1_min']):.3f} | "
              f"BRF({n_trees})={np.mean(results['BRF_f1_min']):.3f}")
    return results

# ── Lancement ────────────────────────────────────────────────
results_ecoli = run_experiment_v2(
    X, y, f"Ecoli ({TARGET_CLASS} vs reste)",
    n_trees=50, k=K_BIRF, n_splits=10
)

display_results(results_ecoli, f"Ecoli — {TARGET_CLASS}=1 (minoritaire)")
statistical_analysis(results_ecoli, f"Ecoli — {TARGET_CLASS}=1 (minoritaire)")

# ── Tableau fold par fold ─────────────────────────────────────
print(f"\n{'='*60}")
print("  Valeurs fold par fold — F1_min")
print(f"{'='*60}")
methods_list = ["BRF", "MPRF", "OBP", "IBEP"]
print(f"{'Fold':<6} " + " ".join(f"{m:>8}" for m in methods_list))
print("-" * 42)
for i in range(10):
    row = f"{i+1:<6} "
    for m in methods_list:
        row += f"  {results_ecoli[f'{m}_f1_min'][i]:.3f}  "
    print(row)
print("-" * 42)
print(f"{'Moy':<6} ", end="")
for m in methods_list:
    print(f"  {np.mean(results_ecoli[f'{m}_f1_min']):.3f}  ", end="")
print()
print(f"{'Std':<6} ", end="")
for m in methods_list:
    print(f"  {np.std(results_ecoli[f'{m}_f1_min']):.3f}  ", end="")
print()

Classes originales : {'cp': 143, 'im': 77, 'pp': 52, 'imU': 35, 'om': 20, 'omL': 5, 'imL': 2, 'imS': 2}
Cible : imU vs reste
Distribution : {0: 301, 1: 35}
Shape : (336, 7)  |  Classes : [301  35]  |  IR ≈ 8.6:1
Score combiné : α=0.7 · F1_min + 0.3 · G-mean

Dataset : Ecoli (imU vs reste)
RF balanced : 50 arbres | MPRF : k=10 | Folds : 10
Score : α=0.7·F1_min + 0.3·G-mean
  Fold 1/10... ✅ MPRF(2)=0.667 | OBP(4)=0.000 | IBEP(4)=0.500 | BRF(50)=0.500
  Fold 2/10... ✅ MPRF(2)=0.667 | OBP(4)=0.222 | IBEP(6)=0.536 | BRF(50)=0.536
  Fold 3/10... ✅ MPRF(4)=0.667 | OBP(4)=0.415 | IBEP(1)=0.624 | BRF(50)=0.579
  Fold 4/10... ✅ MPRF(4)=0.625 | OBP(8)=0.499 | IBEP(4)=0.655 | BRF(50)=0.577
  Fold 5/10... ✅ MPRF(10)=0.600 | OBP(24)=0.488 | IBEP(5)=0.639 | BRF(50)=0.551
  Fold 6/10... ✅ MPRF(2)=0.625 | OBP(6)=0.473 | IBEP(4)=0.643 | BRF(50)=0.526
  Fold 7/10... ✅ MPRF(8)=0.631 | OBP(4)=0.477 | IBEP(3)=0.609 | BRF(50)=0.508
  Fold 8/10... ✅ MPRF(10)=0.627 | OBP(12)=0.501 | IBEP(4)=0.607 | BRF(50)=0.5

# Extraction des règles

In [ ]:
# ============================================================
# MPRF — Extraction et Simplification de la Base de Règles
# Dataset : Risk Factors Cervical Cancer
# Pipeline : Extraction → Sélection/Ranking → Condition Pruning → Merging
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
df = pd.read_csv("/content/1. risk_factors_cervical_cancer.csv")
df.replace("?", np.nan, inplace=True)
df = df.apply(pd.to_numeric, errors="coerce")
df.fillna(df.median(numeric_only=True), inplace=True)

TARGET = "Biopsy"
feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols].values
y = df[TARGET].values.astype(int)
feature_names = feature_cols

print(f"Shape : {X.shape}  |  Classes : {np.bincount(y)}  |  IR ≈ {np.bincount(y)[0]/np.bincount(y)[1]:.1f}:1")

# ── Paramètres ───────────────────────────────────────────────
K_BIRF      = 10
ALPHA       = 0.60   # poids F1_min dans score sélection
N_TREES     = 50
N_SPLITS    = 10
MIN_SUPPORT = 0.05   # ← seuil support Stage 1 (fraction des minoritaires couverts)
MIN_PREC    = 0.30   # ← seuil précision Stage 1

# ── Fonctions MPRF (identiques au code principal) ────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)
    return alpha * f1 + (1 - alpha) * gm

def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    n = len(trees)
    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)
    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)
    best_score = combined_score(y_val, ensemble_predict(trees, X_val, selected), alpha)
    while len(selected) < k and remaining:
        best_cand, best_cand_score = None, -np.inf
        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score, best_cand = score, idx
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break
    return selected

# ════════════════════════════════════════════════════════════
# EXTRACTION DE RÈGLES
# ════════════════════════════════════════════════════════════

def extract_rules_from_tree(tree, feature_names, class_label=1):
    """
    Extrait toutes les règles root-to-leaf d'un arbre de décision.
    Retourne uniquement les règles prédisant class_label (minoritaire).
    Chaque règle = liste de conditions + classe prédite.
    """
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != -2 else "undefined"
        for i in tree_.feature
    ]
    rules = []

    def recurse(node, conditions):
        if tree_.feature[node] != -2:
            # Noeud interne
            name      = feature_name[node]
            threshold = tree_.threshold[node]
            # Branche gauche : <= threshold
            recurse(tree_.children_left[node],
                    conditions + [(name, "<=", threshold)])
            # Branche droite : > threshold
            recurse(tree_.children_right[node],
                    conditions + [(name, ">", threshold)])
        else:
            # Feuille
            predicted_class = np.argmax(tree_.value[node])
            if predicted_class == class_label:
                rules.append(list(conditions))

    recurse(0, [])
    return rules

def extract_rules_from_forest(trees, selected_indices, feature_names, class_label=1):
    """Extrait toutes les règles du sous-ensemble d'arbres MPRF."""
    all_rules = []
    for idx in selected_indices:
        tree_rules = extract_rules_from_tree(trees[idx], feature_names, class_label)
        all_rules.extend(tree_rules)
    return all_rules

def rule_covers(rule, X):
    """Retourne un masque booléen des instances couvertes par la règle."""
    mask = np.ones(len(X), dtype=bool)
    for (feat, op, thresh) in rule:
        feat_idx = feature_names.index(feat)
        if op == "<=":
            mask &= (X[:, feat_idx] <= thresh)
        else:
            mask &= (X[:, feat_idx] > thresh)
    return mask

def rule_stats(rule, X, y):
    """Calcule support et précision d'une règle sur (X, y)."""
    mask    = rule_covers(rule, X)
    covered = np.sum(mask)
    if covered == 0:
        return 0.0, 0.0, 0
    minority_covered = np.sum(y[mask] == 1)
    total_minority   = np.sum(y == 1)
    support   = minority_covered / total_minority if total_minority > 0 else 0.0
    precision = minority_covered / covered
    return support, precision, covered

def rules_to_str(rules, feature_names):
    """Affiche les règles de façon lisible."""
    result = []
    for i, rule in enumerate(rules):
        conditions = " AND ".join(
            [f"{feat} {op} {thresh:.3f}" for (feat, op, thresh) in rule]
        )
        result.append(f"Rule {i+1:>3} : IF {conditions} THEN Biopsy=1")
    return "\n".join(result)

# ════════════════════════════════════════════════════════════
# STAGE 1 — RULE SELECTION & RANKING
# ════════════════════════════════════════════════════════════

def stage1_selection(rules, X_val, y_val,
                     min_support=MIN_SUPPORT, min_prec=MIN_PREC):
    """
    Filtre les règles selon support et précision sur le val set.
    Retourne les règles retenues triées par précision décroissante.
    """
    scored = []
    for rule in rules:
        support, precision, covered = rule_stats(rule, X_val, y_val)
        if support >= min_support and precision >= min_prec:
            scored.append((rule, support, precision, covered))

    # Trier par précision décroissante
    scored.sort(key=lambda x: x[2], reverse=True)
    retained = [r[0] for r in scored]
    return retained, scored

# ════════════════════════════════════════════════════════════
# STAGE 2 — CONDITION PRUNING
# ════════════════════════════════════════════════════════════

def stage2_condition_pruning(rules, X_val, y_val):
    """
    Pour chaque règle, supprime itérativement les conditions
    dont la suppression ne dégrade pas la précision sur le val set.
    """
    pruned_rules = []
    for rule in rules:
        _, original_prec, _ = rule_stats(rule, X_val, y_val)
        current_rule = list(rule)
        changed = True
        while changed:
            changed = False
            for i in range(len(current_rule)):
                candidate = current_rule[:i] + current_rule[i+1:]
                if len(candidate) == 0:
                    break
                _, cand_prec, cand_covered = rule_stats(candidate, X_val, y_val)
                if cand_covered > 0 and cand_prec >= original_prec:
                    current_rule = candidate
                    original_prec = cand_prec
                    changed = True
                    break
        pruned_rules.append(current_rule)

    # Dédupliquer
    unique = []
    seen   = set()
    for rule in pruned_rules:
        key = tuple(sorted([(f, o, round(t, 4)) for (f, o, t) in rule]))
        if key not in seen:
            seen.add(key)
            unique.append(rule)
    return unique

# ════════════════════════════════════════════════════════════
# STAGE 3 — RULE MERGING
# ════════════════════════════════════════════════════════════

def stage3_merging(rules):
    """
    Fusionne les règles qui partagent les mêmes features et opérateurs
    en généralisant les seuils numériques (max pour <=, min pour >).
    """
    def rule_signature(rule):
        return tuple(sorted([(feat, op) for (feat, op, _) in rule]))

    groups = defaultdict(list)
    for rule in rules:
        sig = rule_signature(rule)
        groups[sig].append(rule)

    merged = []
    for sig, group in groups.items():
        if len(group) == 1:
            merged.append(group[0])
        else:
            # Fusionner : pour <=, prendre le max du seuil (plus permissif)
            #             pour  >, prendre le min du seuil (plus permissif)
            merged_rule = []
            for (feat, op) in sig:
                thresholds = [t for rule in group
                              for (f, o, t) in rule if f == feat and o == op]
                if op == "<=":
                    thresh = max(thresholds)
                else:
                    thresh = min(thresholds)
                merged_rule.append((feat, op, thresh))
            merged.append(merged_rule)

    return merged

# ════════════════════════════════════════════════════════════
# PIPELINE PRINCIPAL
# ════════════════════════════════════════════════════════════

def run_rule_extraction_pipeline(X, y, feature_names,
                                  n_trees=N_TREES, k=K_BIRF,
                                  n_splits=N_SPLITS,
                                  min_support=MIN_SUPPORT,
                                  min_prec=MIN_PREC):
    """
    Lance l'expérience MPRF sur les 10 folds et pour chaque fold :
    1. Entraîne le pool RF balanced
    2. Sélectionne le sous-ensemble MPRF
    3. Extrait les règles brutes
    4. Applique le pipeline de simplification en 3 étapes
    5. Affiche l'évolution de la base de règles
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Pour stocker les stats par fold
    stats = defaultdict(list)

    print(f"\n{'='*70}")
    print(f"  PIPELINE EXTRACTION & SIMPLIFICATION DE RÈGLES — MPRF")
    print(f"  Dataset : Risk Factors Cervical Cancer | k={k} | Folds={n_splits}")
    print(f"  Seuils Stage 1 : support≥{min_support} | précision≥{min_prec}")
    print(f"{'='*70}")

    best_fold_rules = None
    best_fold_score = -np.inf
    best_fold_idx   = -1

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)

        X_val, y_val             = X_train[val_idx], y_train[val_idx]
        X_train_sub, y_train_sub = X_train[train_sub_idx], y_train[train_sub_idx]

        # Pool RF balanced
        rf = RandomForestClassifier(n_estimators=n_trees, class_weight='balanced',
                                    random_state=42, n_jobs=-1)
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        # Sélection MPRF
        selected = MPRF(trees, X_val, y_val, k, fold, ALPHA)
        f1_val   = f1_score(y_val, ensemble_predict(trees, X_val, selected),
                            pos_label=1, zero_division=0)

        # ── Stage 0 : Extraction brute ──────────────────────
        raw_rules = extract_rules_from_forest(trees, selected, feature_names)

        # ── Stage 1 : Sélection & Ranking ───────────────────
        s1_rules, s1_scored = stage1_selection(raw_rules, X_val, y_val,
                                               min_support, min_prec)

        # ── Stage 2 : Condition Pruning ──────────────────────
        s2_rules = stage2_condition_pruning(s1_rules, X_val, y_val)

        # ── Stage 3 : Rule Merging ───────────────────────────
        s3_rules = stage3_merging(s2_rules)

        # Stats
        avg_cond_raw = np.mean([len(r) for r in raw_rules]) if raw_rules else 0
        avg_cond_s2  = np.mean([len(r) for r in s2_rules])  if s2_rules  else 0
        avg_cond_s3  = np.mean([len(r) for r in s3_rules])  if s3_rules  else 0

        stats["fold"].append(fold + 1)
        stats["trees"].append(len(selected))
        stats["raw"].append(len(raw_rules))
        stats["s1"].append(len(s1_rules))
        stats["s2"].append(len(s2_rules))
        stats["s3"].append(len(s3_rules))
        stats["avg_cond_raw"].append(avg_cond_raw)
        stats["avg_cond_s3"].append(avg_cond_s3)

        print(f"  Fold {fold+1:>2} | Arbres={len(selected)} | "
              f"Brut={len(raw_rules):>3} → S1={len(s1_rules):>2} → "
              f"S2={len(s2_rules):>2} → S3={len(s3_rules):>2} | "
              f"Cond moy : {avg_cond_raw:.1f}→{avg_cond_s3:.1f} | "
              f"F1_val={f1_val:.3f}")

        # Garder le meilleur fold pour affichage détaillé
        if f1_val > best_fold_score:
            best_fold_score = f1_val
            best_fold_rules = s3_rules
            best_fold_idx   = fold + 1

    # ── Tableau récapitulatif ────────────────────────────────
    print(f"\n{'='*70}")
    print("  TABLEAU RÉCAPITULATIF — Évolution de la base de règles")
    print(f"{'='*70}")
    print(f"  {'Étape':<35} {'Moy':>6} {'Std':>6} {'Min':>5} {'Max':>5}")
    print(f"  {'-'*57}")
    print(f"  {'Arbres sélectionnés (MPRF)':<35} "
          f"{np.mean(stats['trees']):>6.1f} {np.std(stats['trees']):>6.1f} "
          f"{np.min(stats['trees']):>5} {np.max(stats['trees']):>5}")
    print(f"  {'Règles brutes (Stage 0)':<35} "
          f"{np.mean(stats['raw']):>6.1f} {np.std(stats['raw']):>6.1f} "
          f"{np.min(stats['raw']):>5} {np.max(stats['raw']):>5}")
    print(f"  {'Après Sélection/Ranking (Stage 1)':<35} "
          f"{np.mean(stats['s1']):>6.1f} {np.std(stats['s1']):>6.1f} "
          f"{np.min(stats['s1']):>5} {np.max(stats['s1']):>5}")
    print(f"  {'Après Condition Pruning (Stage 2)':<35} "
          f"{np.mean(stats['s2']):>6.1f} {np.std(stats['s2']):>6.1f} "
          f"{np.min(stats['s2']):>5} {np.max(stats['s2']):>5}")
    print(f"  {'Après Rule Merging (Stage 3)':<35} "
          f"{np.mean(stats['s3']):>6.1f} {np.std(stats['s3']):>6.1f} "
          f"{np.min(stats['s3']):>5} {np.max(stats['s3']):>5}")
    print(f"  {'Conditions moy/règle (brut)':<35} "
          f"{np.mean(stats['avg_cond_raw']):>6.1f}")
    print(f"  {'Conditions moy/règle (Stage 3)':<35} "
          f"{np.mean(stats['avg_cond_s3']):>6.1f}")

    compression = (1 - np.mean(stats['s3']) / np.mean(stats['raw'])) * 100 \
                  if np.mean(stats['raw']) > 0 else 0
    print(f"\n  Taux de compression global : {compression:.1f}%")
    print(f"{'='*70}")

    # ── Affichage des règles finales du meilleur fold ────────
    print(f"\n{'='*70}")
    print(f"  RÈGLES FINALES — Meilleur fold (Fold {best_fold_idx}, "
          f"F1_val={best_fold_score:.3f})")
    print(f"  {len(best_fold_rules)} règles diagnostiques — Biopsy=1 (cancer du col)")
    print(f"{'='*70}")
    if best_fold_rules:
        print(rules_to_str(best_fold_rules, feature_names))
    else:
        print("  Aucune règle retenue pour ce fold.")
    print(f"{'='*70}")

    return stats, best_fold_rules

# ── Lancement ────────────────────────────────────────────────
stats, final_rules = run_rule_extraction_pipeline(
    X, y, feature_names,
    n_trees=N_TREES,
    k=K_BIRF,
    n_splits=N_SPLITS,
    min_support=MIN_SUPPORT,
    min_prec=MIN_PREC
)

Shape : (858, 35)  |  Classes : [803  55]  |  IR ≈ 14.6:1

  PIPELINE EXTRACTION & SIMPLIFICATION DE RÈGLES — MPRF
  Dataset : Risk Factors Cervical Cancer | k=10 | Folds=10
  Seuils Stage 1 : support≥0.05 | précision≥0.3
  Fold  1 | Arbres=10 | Brut=138 → S1=43 → S2=36 → S3=31 | Cond moy : 7.5→2.5 | F1_val=0.762
  Fold  2 | Arbres=2 | Brut= 37 → S1= 9 → S2= 9 → S3= 9 | Cond moy : 7.5→2.2 | F1_val=0.727
  Fold  3 | Arbres=2 | Brut= 22 → S1=10 → S2= 8 → S3= 8 | Cond moy : 7.3→2.0 | F1_val=0.800
  Fold  4 | Arbres=4 | Brut= 53 → S1=23 → S2=18 → S3=16 | Cond moy : 7.2→2.3 | F1_val=0.857
  Fold  5 | Arbres=10 | Brut=180 → S1=17 → S2=16 → S3=15 | Cond moy : 7.9→2.5 | F1_val=0.750
  Fold  6 | Arbres=5 | Brut= 68 → S1=23 → S2=22 → S3=22 | Cond moy : 7.1→2.9 | F1_val=0.870
  Fold  7 | Arbres=10 | Brut=144 → S1=33 → S2=32 → S3=30 | Cond moy : 7.9→2.8 | F1_val=0.842
  Fold  8 | Arbres=4 | Brut= 62 → S1=11 → S2=11 → S3=11 | Cond moy : 8.2→2.6 | F1_val=0.842
  Fold  9 | Arbres=4 | Brut= 51 → S1=12

# extraction + evaluation

In [ ]:
# ============================================================
# MPRF — Extraction, Simplification et Évaluation de Règles
# Dataset : Risk Factors Cervical Cancer
# Pipeline : Extraction → S1 → S2 → S3 + Métriques complètes
# Google Colab — copier-coller directement
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score
import warnings
warnings.filterwarnings("ignore")

# ── Chargement ───────────────────────────────────────────────
df = pd.read_csv("/content/1. risk_factors_cervical_cancer.csv")
df.replace("?", np.nan, inplace=True)
df = df.apply(pd.to_numeric, errors="coerce")
df.fillna(df.median(numeric_only=True), inplace=True)

TARGET = "Biopsy"
feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols].values
y = df[TARGET].values.astype(int)
feature_names = feature_cols

print(f"Shape : {X.shape}  |  Classes : {np.bincount(y)}  |  "
      f"IR ≈ {np.bincount(y)[0]/np.bincount(y)[1]:.1f}:1")

# ── Paramètres ───────────────────────────────────────────────
K_BIRF      = 10
ALPHA       = 0.60
N_TREES     = 50
N_SPLITS    = 10
MIN_SUPPORT = 0.05   # ← MODIFIER ICI — seuil support Stage 1
MIN_PREC    = 0.30   # ← MODIFIER ICI — seuil précision Stage 1

# ════════════════════════════════════════════════════════════
# FONCTIONS MPRF
# ════════════════════════════════════════════════════════════

def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def combined_score(y_true, y_pred, alpha):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm = np.sqrt(recall_1 * recall_0)
    return alpha * f1 + (1 - alpha) * gm

def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    n = len(trees)
    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)
    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)
    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected), alpha)
    while len(selected) < k and remaining:
        best_cand, best_cand_score = None, -np.inf
        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score, best_cand = score, idx
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break
    return selected

# ════════════════════════════════════════════════════════════
# EXTRACTION DE RÈGLES
# ════════════════════════════════════════════════════════════

def extract_rules_from_tree(tree, feature_names, class_label=1):
    tree_  = tree.tree_
    feat_n = [feature_names[i] if i != -2 else "undefined"
              for i in tree_.feature]
    rules  = []

    def recurse(node, conditions):
        if tree_.feature[node] != -2:
            name  = feat_n[node]
            thresh = tree_.threshold[node]
            recurse(tree_.children_left[node],
                    conditions + [(name, "<=", thresh)])
            recurse(tree_.children_right[node],
                    conditions + [(name, ">", thresh)])
        else:
            if np.argmax(tree_.value[node]) == class_label:
                rules.append(list(conditions))

    recurse(0, [])
    return rules

def extract_rules_from_forest(trees, selected_indices, feature_names):
    all_rules = []
    for idx in selected_indices:
        all_rules.extend(
            extract_rules_from_tree(trees[idx], feature_names)
        )
    return all_rules

def rule_covers(rule, X):
    mask = np.ones(len(X), dtype=bool)
    for (feat, op, thresh) in rule:
        feat_idx = feature_names.index(feat)
        if op == "<=":
            mask &= (X[:, feat_idx] <= thresh)
        else:
            mask &= (X[:, feat_idx] > thresh)
    return mask

def rule_stats(rule, X, y):
    mask             = rule_covers(rule, X)
    covered          = np.sum(mask)
    if covered == 0:
        return 0.0, 0.0, 0
    minority_covered = np.sum(y[mask] == 1)
    total_minority   = np.sum(y == 1)
    support   = minority_covered / total_minority if total_minority > 0 else 0.0
    precision = minority_covered / covered
    return support, precision, covered

# ════════════════════════════════════════════════════════════
# MÉTRIQUES D'ÉVALUATION DE LA BASE DE RÈGLES
# ════════════════════════════════════════════════════════════

def evaluate_ruleset(rules, X, y, stage_name=""):
    """
    Évalue une base de règles sur (X, y) avec les métriques suivantes :
    - Nombre de règles
    - Conditions moyennes par règle
    - Couverture globale : fraction des minoritaires couverts par ≥1 règle
    - Précision moyenne : moyenne des précisions individuelles
    - Lift moyen : précision_règle / taux_base_minoritaire
    - Overlap moyen : nombre moyen de règles couvrant chaque instance minoritaire
    - Complexité : nb_règles × conditions_moyennes
    """
    if len(rules) == 0:
        return {
            "stage"         : stage_name,
            "n_rules"       : 0,
            "avg_conditions": 0.0,
            "coverage"      : 0.0,
            "avg_precision" : 0.0,
            "avg_lift"      : 0.0,
            "avg_overlap"   : 0.0,
            "complexity"    : 0.0,
        }

    base_rate    = np.sum(y == 1) / len(y)
    total_min    = np.sum(y == 1)
    minority_idx = np.where(y == 1)[0]

    # Précision, lift, support par règle
    precisions = []
    supports   = []
    lifts      = []
    n_conds    = []

    for rule in rules:
        sup, prec, cov = rule_stats(rule, X, y)
        precisions.append(prec)
        supports.append(sup)
        lifts.append(prec / base_rate if base_rate > 0 else 0.0)
        n_conds.append(len(rule))

    # Couverture globale : union des instances minoritaires couvertes
    covered_minority = np.zeros(len(X), dtype=bool)
    for rule in rules:
        covered_minority |= rule_covers(rule, X)
    coverage = np.sum(covered_minority[minority_idx]) / total_min \
               if total_min > 0 else 0.0

    # Overlap : pour chaque instance minoritaire, combien de règles la couvrent
    overlap_counts = np.zeros(total_min)
    for rule in rules:
        mask = rule_covers(rule, X)
        for i, idx in enumerate(minority_idx):
            if mask[idx]:
                overlap_counts[i] += 1
    avg_overlap = np.mean(overlap_counts[overlap_counts > 0]) \
                  if np.any(overlap_counts > 0) else 0.0

    avg_conditions = np.mean(n_conds)
    complexity     = len(rules) * avg_conditions

    return {
        "stage"         : stage_name,
        "n_rules"       : len(rules),
        "avg_conditions": round(avg_conditions, 2),
        "coverage"      : round(coverage, 3),
        "avg_precision" : round(np.mean(precisions), 3),
        "avg_lift"      : round(np.mean(lifts), 3),
        "avg_overlap"   : round(avg_overlap, 2),
        "complexity"    : round(complexity, 1),
    }

def print_metrics_table(metrics_list):
    """Affiche le tableau de comparaison des métriques par stage."""
    header = (f"  {'Étape':<30} {'Règles':>7} {'Cond/R':>7} "
              f"{'Couv.':>7} {'Préc.':>7} {'Lift':>7} "
              f"{'Overlap':>8} {'Compl.':>8}")
    print(header)
    print(f"  {'-'*76}")
    for m in metrics_list:
        print(f"  {m['stage']:<30} {m['n_rules']:>7} {m['avg_conditions']:>7.1f} "
              f"{m['coverage']:>7.3f} {m['avg_precision']:>7.3f} "
              f"{m['avg_lift']:>7.2f} {m['avg_overlap']:>8.2f} "
              f"{m['complexity']:>8.1f}")

# ════════════════════════════════════════════════════════════
# STAGES DE SIMPLIFICATION
# ════════════════════════════════════════════════════════════

def stage1_selection(rules, X_val, y_val,
                     min_support=MIN_SUPPORT, min_prec=MIN_PREC):
    scored = []
    for rule in rules:
        support, precision, covered = rule_stats(rule, X_val, y_val)
        if support >= min_support and precision >= min_prec:
            scored.append((rule, support, precision))
    scored.sort(key=lambda x: x[2], reverse=True)
    return [r[0] for r in scored], scored

def stage2_condition_pruning(rules, X_val, y_val):
    pruned_rules = []
    for rule in rules:
        _, original_prec, _ = rule_stats(rule, X_val, y_val)
        current_rule = list(rule)
        changed = True
        while changed:
            changed = False
            for i in range(len(current_rule)):
                candidate = current_rule[:i] + current_rule[i+1:]
                if len(candidate) == 0:
                    break
                _, cand_prec, cand_covered = rule_stats(candidate, X_val, y_val)
                if cand_covered > 0 and cand_prec >= original_prec:
                    current_rule  = candidate
                    original_prec = cand_prec
                    changed = True
                    break
        pruned_rules.append(current_rule)
    # Dédupliquer
    unique, seen = [], set()
    for rule in pruned_rules:
        key = tuple(sorted([(f, o, round(t, 4)) for (f, o, t) in rule]))
        if key not in seen:
            seen.add(key)
            unique.append(rule)
    return unique

def stage3_merging(rules):
    def rule_signature(rule):
        return tuple(sorted([(feat, op) for (feat, op, _) in rule]))

    groups = defaultdict(list)
    for rule in rules:
        groups[rule_signature(rule)].append(rule)

    merged = []
    for sig, group in groups.items():
        if len(group) == 1:
            merged.append(group[0])
        else:
            merged_rule = []
            for (feat, op) in sig:
                thresholds = [t for r in group
                              for (f, o, t) in r if f == feat and o == op]
                thresh = max(thresholds) if op == "<=" else min(thresholds)
                merged_rule.append((feat, op, thresh))
            merged.append(merged_rule)
    return merged

def rules_to_str(rules):
    result = []
    for i, rule in enumerate(rules):
        conds = " AND ".join(
            [f"{feat} {op} {thresh:.3f}" for (feat, op, thresh) in rule]
        )
        result.append(f"  Rule {i+1:>2} : IF {conds} THEN Biopsy=1")
    return "\n".join(result)

# ════════════════════════════════════════════════════════════
# PIPELINE PRINCIPAL
# ════════════════════════════════════════════════════════════

def run_pipeline(X, y, feature_names):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

    # Accumulateurs métriques par stage (sur les 10 folds)
    accum = {s: defaultdict(list)
             for s in ["raw", "s1", "s2", "s3"]}

    best_fold_rules = None
    best_fold_score = -np.inf
    best_fold_idx   = -1
    best_fold_X_val = None
    best_fold_y_val = None

    print(f"\n{'='*78}")
    print(f"  PIPELINE EXTRACTION & SIMPLIFICATION — MPRF")
    print(f"  Support≥{MIN_SUPPORT} | Précision≥{MIN_PREC} | k={K_BIRF}")
    print(f"{'='*78}")
    print(f"  {'Fold':>4} | {'Arbres':>6} | {'Brut':>5} → {'S1':>3} → "
          f"{'S2':>3} → {'S3':>3} | {'Cov_brut':>8} {'Cov_S3':>7} | "
          f"{'F1_val':>6}")
    print(f"  {'-'*72}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        val_size      = int(len(train_idx) * 0.2)
        rng_split     = np.random.RandomState(fold)
        val_idx       = rng_split.choice(len(y_train), val_size, replace=False)
        train_sub_idx = np.setdiff1d(np.arange(len(y_train)), val_idx)

        X_val, y_val             = X_train[val_idx], y_train[val_idx]
        X_train_sub, y_train_sub = X_train[train_sub_idx], y_train[train_sub_idx]

        rf = RandomForestClassifier(n_estimators=N_TREES,
                                    class_weight='balanced',
                                    random_state=42, n_jobs=-1)
        rf.fit(X_train_sub, y_train_sub)
        trees = rf.estimators_

        selected = MPRF(trees, X_val, y_val, K_BIRF, fold, ALPHA)
        f1_val   = f1_score(y_val,
                            ensemble_predict(trees, X_val, selected),
                            pos_label=1, zero_division=0)

        # Extraction et simplification
        raw_rules = extract_rules_from_forest(trees, selected, feature_names)
        s1_rules, _ = stage1_selection(raw_rules, X_val, y_val)
        s2_rules    = stage2_condition_pruning(s1_rules, X_val, y_val)
        s3_rules    = stage3_merging(s2_rules)

        # Évaluation sur le val set
        m_raw = evaluate_ruleset(raw_rules, X_val, y_val, "Brut (Stage 0)")
        m_s1  = evaluate_ruleset(s1_rules,  X_val, y_val, "Sélection (Stage 1)")
        m_s2  = evaluate_ruleset(s2_rules,  X_val, y_val, "Cond. Pruning (Stage 2)")
        m_s3  = evaluate_ruleset(s3_rules,  X_val, y_val, "Merging (Stage 3)")

        for key in ["n_rules","avg_conditions","coverage","avg_precision",
                    "avg_lift","avg_overlap","complexity"]:
            accum["raw"][key].append(m_raw[key])
            accum["s1"][key].append(m_s1[key])
            accum["s2"][key].append(m_s2[key])
            accum["s3"][key].append(m_s3[key])

        print(f"  {fold+1:>4} | {len(selected):>6} | {len(raw_rules):>5} → "
              f"{len(s1_rules):>3} → {len(s2_rules):>3} → {len(s3_rules):>3} | "
              f"{m_raw['coverage']:>8.3f} {m_s3['coverage']:>7.3f} | "
              f"{f1_val:>6.3f}")

        if f1_val > best_fold_score:
            best_fold_score = f1_val
            best_fold_rules = s3_rules
            best_fold_idx   = fold + 1
            best_fold_X_val = X_val
            best_fold_y_val = y_val

    # ── Tableau récapitulatif des métriques ──────────────────
    print(f"\n{'='*78}")
    print("  MÉTRIQUES MOYENNES SUR 10 FOLDS — Comparaison par stage")
    print(f"{'='*78}")
    avg_metrics = []
    for stage_key, stage_name in [
        ("raw", "Brut (Stage 0)"),
        ("s1",  "Sélection (Stage 1)"),
        ("s2",  "Cond. Pruning (Stage 2)"),
        ("s3",  "Merging (Stage 3)"),
    ]:
        avg_metrics.append({
            "stage"         : stage_name,
            "n_rules"       : round(np.mean(accum[stage_key]["n_rules"]), 1),
            "avg_conditions": round(np.mean(accum[stage_key]["avg_conditions"]), 2),
            "coverage"      : round(np.mean(accum[stage_key]["coverage"]), 3),
            "avg_precision" : round(np.mean(accum[stage_key]["avg_precision"]), 3),
            "avg_lift"      : round(np.mean(accum[stage_key]["avg_lift"]), 2),
            "avg_overlap"   : round(np.mean(accum[stage_key]["avg_overlap"]), 2),
            "complexity"    : round(np.mean(accum[stage_key]["complexity"]), 1),
        })
    print_metrics_table(avg_metrics)

    # ── Compression ──────────────────────────────────────────
    raw_mean = np.mean(accum["raw"]["n_rules"])
    s3_mean  = np.mean(accum["s3"]["n_rules"])
    comp_rules = (1 - s3_mean / raw_mean) * 100 if raw_mean > 0 else 0
    raw_cond  = np.mean(accum["raw"]["avg_conditions"])
    s3_cond   = np.mean(accum["s3"]["avg_conditions"])
    comp_cond = (1 - s3_cond / raw_cond) * 100 if raw_cond > 0 else 0
    cov_raw   = np.mean(accum["raw"]["coverage"])
    cov_s3    = np.mean(accum["s3"]["coverage"])
    cov_loss  = (cov_raw - cov_s3) * 100

    print(f"\n  Compression règles    : {raw_mean:.1f} → {s3_mean:.1f} "
          f"({comp_rules:.1f}% réduction)")
    print(f"  Compression conditions: {raw_cond:.1f} → {s3_cond:.1f} "
          f"({comp_cond:.1f}% réduction)")
    print(f"  Perte couverture      : {cov_raw:.3f} → {cov_s3:.3f} "
          f"({cov_loss:.1f}% de perte)")
    print(f"  Précision maintenue   : "
          f"{np.mean(accum['raw']['avg_precision']):.3f} → "
          f"{np.mean(accum['s3']['avg_precision']):.3f}")

    # ── Règles finales du meilleur fold ──────────────────────
    print(f"\n{'='*78}")
    print(f"  RÈGLES FINALES — Meilleur fold (Fold {best_fold_idx}, "
          f"F1_val={best_fold_score:.3f})")
    print(f"  {len(best_fold_rules)} règles — Biopsy=1 (cancer du col cervical)")
    print(f"{'='*78}")
    print(rules_to_str(best_fold_rules))

    # Métriques détaillées des règles finales
    print(f"\n  {'Rule':<6} {'Support':>8} {'Précision':>10} "
          f"{'Lift':>7} {'Conditions':>11}")
    print(f"  {'-'*44}")
    base_rate = np.sum(best_fold_y_val == 1) / len(best_fold_y_val)
    for i, rule in enumerate(best_fold_rules):
        sup, prec, cov = rule_stats(rule, best_fold_X_val, best_fold_y_val)
        lift = prec / base_rate if base_rate > 0 else 0.0
        print(f"  Rule {i+1:<2} {sup:>8.3f} {prec:>10.3f} "
              f"{lift:>7.2f} {len(rule):>11}")
    print(f"{'='*78}")

    return accum, best_fold_rules

# ── Lancement ────────────────────────────────────────────────
accum, final_rules = run_pipeline(X, y, feature_names)

Shape : (858, 35)  |  Classes : [803  55]  |  IR ≈ 14.6:1

  PIPELINE EXTRACTION & SIMPLIFICATION — MPRF
  Support≥0.05 | Précision≥0.3 | k=10
  Fold | Arbres |  Brut →  S1 →  S2 →  S3 | Cov_brut  Cov_S3 | F1_val
  ------------------------------------------------------------------------
     1 |     10 |   138 →  43 →  36 →  31 |    0.818   0.818 |  0.762
     2 |      2 |    37 →   9 →   9 →   9 |    0.889   0.889 |  0.727
     3 |      2 |    22 →  10 →   8 →   8 |    0.800   0.950 |  0.800
     4 |      4 |    53 →  23 →  18 →  16 |    0.929   0.929 |  0.857
     5 |     10 |   180 →  17 →  16 →  15 |    1.000   1.000 |  0.750
     6 |      5 |    68 →  23 →  22 →  22 |    1.000   1.000 |  0.870
     7 |     10 |   144 →  33 →  32 →  30 |    0.889   0.889 |  0.842
     8 |      4 |    62 →  11 →  11 →  11 |    0.889   0.889 |  0.842
     9 |      4 |    51 →  12 →  10 →  10 |    0.900   0.900 |  0.762
    10 |      4 |    48 →  11 →   9 →   9 |    1.000   1.000 |  0.900

  MÉTRIQUES